# Memory and Context Engineering for AI Agents with Oracle AI Database, Langchain and Tavily

[![Open in Colab](https://img.shields.io/badge/Open%20in-Colab-F9AB00?style=flat-square&logo=googlecolab)](https://colab.research.google.com/github/oracle-devrel/oracle-ai-developer-hub/blob/main/notebooks/memory_context_engineering_agents.ipynb)

--------



In this notebook, you'll learn how to engineer memory systems that give AI agents the ability to remember, learn, and adapt across conversations. 
Moving beyond simple RAG, we implement a complete **Memory Manager** with six distinct memory types—each serving a specific cognitive function.



## What You'll Build

| Memory Type | Purpose | Storage |
|-------------|---------|---------|
| **Conversational** | Chat history per thread | SQL Table |
| **Knowledge Base** | Searchable documents & facts | Vector-Enabled SQL Table |
| **Workflow** | Learned action patterns | Vector-Enabled SQL Table |
| **Toolbox** | Dynamic tool definitions | Vector-Enabled SQL Table |
| **Entity** | People, places, systems extracted from context | Vector-Enabled SQL Table |
| **Summary** | Compressed context for long conversations | Vector-Enabled SQL Table |


## Key Concepts Covered

- **Memory Engineering**: Design patterns for agent memory systems
- **Context Engineering**: Techniques for optimizing what goes into the LLM context
- **Context Window Management**: Monitor usage, auto-summarize at thresholds
- **Just-in-Time Retrieval**: Compact summaries with on-demand expansion
- **Dynamic Tool Calling**: Semantic tool discovery and execution
- **Entity Extraction**: LLM-powered entity recognition and storage



## Prerequisites

- Python 3.10+
- Oracle AI Database (local Docker or cloud)
- OpenAI API key
- Tavily API key

## By the End
You'll have a reusable `MemoryLayer` class and agent loop that demonstrates how modern AI agents maintain context, learn from interactions, and manage information across sessions.

In [1]:
# Previous install command kept for reference:
# ! pip install -qU langchain-oracledb sentence-transformers langchain-openai langchain tavily-python datasets
! pip install -qU langchain-oracledb sentence-transformers langchain-openai langchain langchain-community langchain-huggingface tavily-python datasets


[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# Local Installation of Oracle AI Database via Docker [Memory Core]

--------

This section walks you through setting up **Oracle AI Database 26ai** locally using Docker. Oracle AI Database is a converged database that combines relational, document, graph, and vector data in a single engine—making it ideal for AI applications that need semantic search, embeddings storage, and vector similarity queries.

**What you'll do:**
1. Pull and run the Oracle Database Docker container
2. Establish a connection from Python using `oracledb`
3. Create a dedicated user for vector operations

This local setup gives you a fully functional Oracle database for development and testing without needing cloud infrastructure.

### Installing Oracle AI Database via Docker

For this notebook we will be using a local installation of [Oracle AI Database](https://www.oracle.com/database/free/get-started/)

1. Install & start Docker. Docker Desktop (Mac/Windows) or Docker Engine (Linux). Make sure it’s running.
    - If installed with Docker Enginer, run from terminal ```open /Applications/Docker.app```
2. We are going to pull the [docker image](https://container-registry.oracle.com/ords/f?p=113:4:13936724845291:::4:P4_REPOSITORY,AI_REPOSITORY,AI_REPOSITORY_NAME,P4_REPOSITORY_NAME,P4_EULA_ID,P4_BUSINESS_AREA_ID:1863,1863,Oracle%20Database%20Free,Oracle%20Database%20Free,1,0&cs=3cVNH02fFYhB723ODpNnr0JZI1S7Z64nRyL_zC1Ls5BSVLafGsOLMFvFoPhn8JeeB8tXPhkfFKH8-dkrL_z3_0g)
3. Run a container with oracle image

    ```
      docker run -d \
        --name oracle-free \
        -p 1521:1521 -p 5500:5500 \
        -e ORACLE_PWD=OraclePwd_2025 \
        -v $HOME/oracle/full_data:/opt/oracle/oradata \
        container-registry.oracle.com/database/free:latest

    ```

> 🚫 **Troubleshoot**  
> If you see the error:  
> *`docker: Error response from daemon: Conflict. The container name "/oracle-full" is already in use by container ... You have to remove (or rename) that container to be able to reuse that name.`*  
>
> 🧩 **Fix:**  
> - Remove the existing container:  
>   ```bash
>   docker rm oracle-free
>   ```  
> - Then re-run your Docker command from **Step 3** to start a new container.


### 🚀 One-Click Database Setup

The cell below handles **everything automatically**:
- ✅ Checks if Docker is running
- ✅ Checks if Oracle container exists and is healthy
- ✅ Waits for database to be ready (with progress indicator)
- ✅ Fixes the listener for ARM Macs (Apple Silicon)
- ✅ Creates the VECTOR user with proper privileges
- ✅ Tests the connection

**Just run the cell below and wait for the ✅ success message!**


In [2]:
import subprocess
import time
import sys

def setup_oracle_database(container_name="oracle-free", vector_password="VectorPwd_2025"):
    """
    Complete Oracle Database setup - handles everything in one call.
    
    This function:
    1. Checks Docker is running
    2. Verifies container exists and is healthy
    3. Waits for database to be ready
    4. Fixes listener for ARM Macs
    5. Creates VECTOR user
    6. Tests connection
    """
    print("=" * 60)
    print("🚀 ORACLE DATABASE SETUP")
    print("=" * 60)
    
    # Step 1: Check Docker
    print("\n[1/6] Checking Docker...")
    try:
        result = subprocess.run(['docker', 'info'], capture_output=True, text=True, timeout=10)
        if result.returncode != 0:
            print("   ❌ Docker is not running!")
            print("   💡 Start Docker Desktop and try again.")
            return False
        print("   ✅ Docker is running")
    except FileNotFoundError:
        print("   ❌ Docker not found! Please install Docker.")
        return False
    except subprocess.TimeoutExpired:
        print("   ❌ Docker is not responding. Please restart Docker.")
        return False
    
    # Step 2: Check container
    print(f"\n[2/6] Checking container '{container_name}'...")
    result = subprocess.run(
        ['docker', 'ps', '-a', '--filter', f'name={container_name}', '--format', '{{.Status}}'],
        capture_output=True, text=True
    )
    status = result.stdout.strip()
    
    if not status:
        print(f"   ❌ Container '{container_name}' not found!")
        print("   💡 Run the docker run command from the previous cell first.")
        return False
    elif "Up" not in status:
        print(f"   ⚠️  Container exists but not running. Starting...")
        subprocess.run(['docker', 'start', container_name], capture_output=True)
        time.sleep(5)
    
    print(f"   ✅ Container is running")
    
    def probe_database_ready():
        """True readiness check via SQL: CDB OPEN and FREEPDB1 READ WRITE."""
        probe_sql = """
SET HEADING OFF FEEDBACK OFF PAGESIZE 0 VERIFY OFF ECHO OFF
WHENEVER SQLERROR EXIT SQL.SQLCODE
SELECT status || ':' || open_mode
FROM v$instance
CROSS JOIN (SELECT open_mode FROM v$pdbs WHERE name = 'FREEPDB1');
EXIT;
"""
        probe = subprocess.run(
            ['docker', 'exec', '-i', container_name, 'bash', '-c',
             'export ORACLE_SID=FREE && sqlplus -s / as sysdba'],
            input=probe_sql,
            capture_output=True,
            text=True
        )

        stdout_lines = [line.strip() for line in probe.stdout.splitlines() if line.strip()]
        normalized = " ".join(stdout_lines).upper()
        is_ready = probe.returncode == 0 and "OPEN:READ WRITE" in normalized

        if stdout_lines:
            details = " | ".join(stdout_lines)
        else:
            details = probe.stderr.strip() or f"sqlplus exited with code {probe.returncode}"

        return is_ready, details

    # Step 3: Wait for database ready
    print("\n[3/6] Waiting for database to be ready...")
    print("   (True check: probing instance state and FREEPDB1 open mode)")

    max_wait = 300  # 5 minutes
    check_interval = 5
    elapsed = 0
    last_details = None

    while elapsed < max_wait:
        # Ensure container is still up while waiting
        status_result = subprocess.run(
            ['docker', 'ps', '--filter', f'name={container_name}', '--format', '{{.Status}}'],
            capture_output=True, text=True
        )
        running_status = status_result.stdout.strip()
        if "up" not in running_status.lower():
            print(f"\n   ❌ Container stopped while waiting: {running_status or 'unknown status'}")
            return False

        ready, details = probe_database_ready()
        if ready:
            print("\n   ✅ Database is ready (OPEN:READ WRITE)")
            break

        # Show probe state when it changes
        if details != last_details:
            print(f"\n   🔎 Probe status: {details}")
            last_details = details

        dots = "." * ((elapsed // check_interval) % 4 + 1)
        print(f"\r   ⏳ Waiting{dots.ljust(5)} ({elapsed}s elapsed)", end="", flush=True)
        time.sleep(check_interval)
        elapsed += check_interval
    else:
        print(f"\n   ❌ Timeout waiting for database. Check 'docker exec -it {container_name} bash'")
        return False
    
    # Step 4: Fix listener (for ARM Macs)
    print("\n[4/6] Configuring listener...")
    
    # Fix listener.ora
    subprocess.run(
        ['docker', 'exec', container_name, 'bash', '-c',
         "sed -i 's/HOST = [^)]*)/HOST = 0.0.0.0)/g' /opt/oracle/product/26ai/dbhomeFree/network/admin/listener.ora"],
        capture_output=True
    )
    
    # Restart listener
    subprocess.run(['docker', 'exec', container_name, 'lsnrctl', 'stop'], capture_output=True)
    start_result = subprocess.run(
        ['docker', 'exec', container_name, 'lsnrctl', 'start'],
        capture_output=True, text=True
    )
    
    if "Listening on" not in start_result.stdout:
        print("   ❌ Failed to start listener")
        return False
    
    # Register services
    subprocess.run(
        ['docker', 'exec', container_name, 'bash', '-c',
         "export ORACLE_SID=FREE && sqlplus -s / as sysdba <<< 'ALTER SYSTEM REGISTER;'"],
        capture_output=True
    )
    print("   ✅ Listener configured and running")
    
    # Step 5: Create VECTOR user
    print("\n[5/6] Creating VECTOR user...")
    
    create_user_sql = f'''
    DECLARE
        user_count NUMBER;
    BEGIN
        SELECT COUNT(*) INTO user_count FROM all_users WHERE username = 'VECTOR';
        IF user_count = 0 THEN
            EXECUTE IMMEDIATE 'CREATE USER VECTOR IDENTIFIED BY {vector_password}';
            EXECUTE IMMEDIATE 'GRANT CONNECT, RESOURCE, CREATE SESSION TO VECTOR';
            EXECUTE IMMEDIATE 'GRANT UNLIMITED TABLESPACE TO VECTOR';
            EXECUTE IMMEDIATE 'GRANT CREATE TABLE, CREATE SEQUENCE, CREATE VIEW TO VECTOR';
            DBMS_OUTPUT.PUT_LINE('CREATED');
        ELSE
            DBMS_OUTPUT.PUT_LINE('EXISTS');
        END IF;
    END;
    /
    '''
    
    result = subprocess.run(
        ['docker', 'exec', container_name, 'bash', '-c',
         f"export ORACLE_SID=FREE && sqlplus -s / as sysdba <<< \"ALTER SESSION SET CONTAINER = FREEPDB1; {create_user_sql}\""],
        capture_output=True, text=True
    )
    
    if "ORA-" in result.stdout:
        print(f"   ⚠️  Warning: {result.stdout}")
    else:
        print("   ✅ VECTOR user ready")
    
    # Step 6: Test connection
    print("\n[6/6] Testing connection...")
    try:
        import oracledb
        conn = oracledb.connect(
            user="VECTOR",
            password=vector_password,
            dsn="127.0.0.1:1521/FREEPDB1"
        )
        with conn.cursor() as cur:
            cur.execute("SELECT 1 FROM dual")
            cur.fetchone()
        conn.close()
        print("   ✅ Connection successful!")
    except Exception as e:
        print(f"   ❌ Connection failed: {e}")
        return False
    
    # Success!
    print("\n" + "=" * 60)
    print("🎉 SETUP COMPLETE!")
    print("=" * 60)
    print(f"""
You can now connect to Oracle:
    User: VECTOR
    Password: {vector_password}
    DSN: 127.0.0.1:1521/FREEPDB1
""")
    return True


In [3]:
# Run this cell after starting your Docker container
# It handles everything: waits for ready, fixes listener, creates user, tests connection
setup_oracle_database()

🚀 ORACLE DATABASE SETUP

[1/6] Checking Docker...
   ✅ Docker is running

[2/6] Checking container 'oracle-free'...
   ✅ Container is running

[3/6] Waiting for database to be ready...
   (True check: probing instance state and FREEPDB1 open mode)

   ✅ Database is ready (OPEN:READ WRITE)

[4/6] Configuring listener...
   ✅ Listener configured and running

[5/6] Creating VECTOR user...
   ✅ VECTOR user ready

[6/6] Testing connection...
   ✅ Connection successful!

🎉 SETUP COMPLETE!

You can now connect to Oracle:
    User: VECTOR
    Password: VectorPwd_2025
    DSN: 127.0.0.1:1521/FREEPDB1



True

In [4]:
def ensure_oracle_vector_memory(container_name="oracle-free", target_size="512M"):
    """Ensure Oracle has vector memory available for HNSW indexes."""
    import subprocess

    def size_to_bytes(size):
        text = str(size).strip().upper()
        if text.isdigit():
            return int(text)

        units = {"K": 1024, "M": 1024**2, "G": 1024**3}
        suffix = text[-1]
        if suffix in units:
            return int(float(text[:-1]) * units[suffix])

        raise ValueError(f"Unsupported size format: {size}")

    def run_sysdba_sql(sql):
        return subprocess.run(
            [
                "docker", "exec", "-i", container_name,
                "bash", "-lc",
                "export ORACLE_SID=FREE; sqlplus -s / as sysdba",
            ],
            input=sql,
            capture_output=True,
            text=True,
        )

    check_sql = """
SET HEADING OFF FEEDBACK OFF PAGESIZE 0 VERIFY OFF ECHO OFF
SELECT value FROM v$parameter WHERE name = 'vector_memory_size';
EXIT;
"""
    result = run_sysdba_sql(check_sql)
    current_value = result.stdout.strip().splitlines()[-1].strip() if result.stdout.strip() else "UNKNOWN"

    if size_to_bytes(current_value) == size_to_bytes(target_size):
        print(f"Vector memory already configured: {target_size}")
        return

    print(f"Configuring vector_memory_size: {current_value} -> {target_size}")
    configure_sql = f"""
WHENEVER SQLERROR EXIT SQL.SQLCODE
ALTER SYSTEM SET vector_memory_size={target_size} SCOPE=SPFILE;
SHUTDOWN IMMEDIATE;
STARTUP;
ALTER PLUGGABLE DATABASE ALL OPEN;
ALTER SYSTEM REGISTER;
EXIT;
"""
    result = run_sysdba_sql(configure_sql)
    if result.returncode != 0:
        raise RuntimeError(result.stderr or result.stdout)

    print(f"Vector memory configured to {target_size}; database restarted and FREEPDB1 reopened.")

ensure_oracle_vector_memory()

Vector memory already configured: 512M


### Connection Helper Function

In the code below we have a reusable function that connects to Oracle Database with automatic retry logic and helpful error messages.

**What it does:**
1. Attempts to connect using the `oracledb` Python driver
2. Retries up to 3 times if the connection fails (useful when the database is still starting)
3. Prints the Oracle version banner on successful connection. This will also include the version you are running
4. Provides troubleshooting hints for common connection errors


In [5]:
import oracledb
import time

def connect_to_oracle(max_retries=3, retry_delay=5, user="sys", password="OraclePwd_2025", dsn="127.0.0.1:1521/FREEPDB1", program="langchain_oracledb_deep_research_demo"):
    """
    Connect to Oracle database with retry logic and better error handling.
    
    Args:
        max_retries: Maximum number of connection attempts
        retry_delay: Seconds to wait between retries
    """
    
    for attempt in range(1, max_retries + 1):
        try:
            print(f"Connection attempt {attempt}/{max_retries}...")
            conn = oracledb.connect(
                user=user,
                password=password,
                dsn=dsn,
                program=program
            )
            print("✓ Connected successfully!")
            
            # Test the connection
            with conn.cursor() as cur:
                cur.execute("SELECT banner FROM v$version WHERE banner LIKE 'Oracle%';")
                banner = cur.fetchone()[0]
                # Banner should include the version you are running
                print(f"\n{banner}")
            
            return conn
            
        except oracledb.OperationalError as e:
            error_msg = str(e)
            print(f"✗ Connection failed (attempt {attempt}/{max_retries})")
            
            if "DPY-4011" in error_msg or "Connection reset by peer" in error_msg:
                print("  → This usually means:")
                print("    1. Database is still starting up (wait 2-3 minutes)")
                print("    2. Listener configuration issue")
                print("    3. Container is not running")
                
                if attempt < max_retries:
                    print(f"\n  Waiting {retry_delay} seconds before retry...")
                    time.sleep(retry_delay)
                else:
                    print("\n  💡 Try running: setup_oracle_database()")
                    print("     This will fix the listener and verify the connection.")
                    raise
            else:
                raise
        except Exception as e:
            print(f"✗ Unexpected error: {e}")
            raise
    
    raise ConnectionError("Failed to connect after all retries")

Ensure you have your Docker Engine running before going through the next steps

Connect as the `VECTOR` user dedicated schema for storing embeddings and vector data.


In [6]:
vector_conn = connect_to_oracle(
    user="VECTOR",
    password="VectorPwd_2025",
    dsn="127.0.0.1:1521/FREEPDB1",
    program="devrel.hub.memory_engineering",
)

print("Using user:", vector_conn.username)

# One-time cleanup: remove prior user-created indexes on the demo vector table.
# This prevents ORA-01408 when rerunning index creation with a new name.
def one_time_cleanup_vector_demo_indexes(conn):
    dropped = []
    with conn.cursor() as cur:
        cur.execute("""
            SELECT index_name
            FROM user_indexes
            WHERE table_name = 'VECTOR_SEARCH_DEMO'
              AND generated = 'N'
        """)
        indexes = [row[0] for row in cur.fetchall()]

        for idx in indexes:
            try:
                cur.execute(f'DROP INDEX "{idx}"')
                dropped.append(idx)
            except Exception as e:
                print(f"  ⚠️ Could not drop index {idx}: {e}")

    conn.commit()
    if dropped:
        print(f"🧹 One-time cleanup: dropped {len(dropped)} old index(es): {', '.join(dropped)}")
    else:
        print("🧹 One-time cleanup: no existing user-created indexes on VECTOR_SEARCH_DEMO")

one_time_cleanup_vector_demo_indexes(vector_conn)


Connection attempt 1/3...
✓ Connected successfully!

Oracle AI Database 26ai Free Release 23.26.2.0.0 - Develop, Learn, and Run for Free
Using user: VECTOR
🧹 One-time cleanup: dropped 1 old index(es): oravs_hnsw


✅ **Setup complete!** You now have Oracle AI Database running locally with an active connection.

Next, we'll create vector-enabled SQL tables using **LangChain's OracleVS integration** to store embeddings and metadata for semantic search.

# Vector Search With Langchain and Oracle AI Database

--------

This section demonstrates how to use **LangChain's OracleVS abstraction** over vector-enabled SQL tables to store and search documents using semantic similarity. 

Vector search enables finding documents based on meaning rather than exact keyword matches.

## What You'll Learn

| Step | Description |
|------|-------------|
| **1. Initialize Embeddings** | Load a HuggingFace embedding model to convert text into vectors |
| **2. Create Vector-Enabled Table (OracleVS)** | Set up an Oracle-backed vector-enabled table with cosine distance |
| **3. Create Index** | Build an HNSW (Hierarchical Navigable Small World) index for fast similarity search |
| **4. Add Documents** | Store text with metadata in the vector database |
| **5. Query** | Search for similar documents using natural language |
| **6. Filter Results** | Use metadata filters to narrow down search results |

## Key Components

- **`OracleVS`**: LangChain abstraction over Oracle vector-enabled SQL tables
- **`HuggingFaceEmbeddings`**: Converts text to 768-dimensional vectors
- **`DistanceStrategy.COSINE`**: Measures vector similarity using cosine distance
- **HNSW Index**: Graph-based ANN index for fast and accurate nearest-neighbor retrieval


## Creating Vector-Enabled Tables with LangChain OracleVS

In [7]:
from langchain_oracledb.vectorstores import OracleVS
# from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_oracledb.vectorstores.oraclevs import create_index
from langchain_community.vectorstores.utils import DistanceStrategy

# Initialize the embedding model
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-mpnet-base-v2"
)

# Initialize the OracleVS handle over a vector-enabled SQL table
vector_store = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name="VECTOR_SEARCH_DEMO",
    distance_strategy=DistanceStrategy.COSINE,
)


C:\Users\fassis\AppData\Local\Temp\ipykernel_57832\4090039570.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores.utils import DistanceStrategy


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [8]:
# Helper to safely create index (skips if already exists)
def safe_create_index(conn, vs, idx_name):
    """Create index, skipping if it already exists."""
    try:
        create_index(
            client=conn,
            vector_store=vs,
            params={"idx_name": idx_name, "idx_type": "HNSW"}
        )
        print(f"  ✅ Created index: {idx_name}")
    except Exception as e:
        if "ORA-00955" in str(e):
            print(f"  ⏭️ Index already exists: {idx_name} (skipped)")
        else:
            raise


In [9]:
import logging

# Suppress langchain_oracledb logging, remove this if you want to see the debug logs
logging.getLogger("langchain_oracledb").setLevel(logging.CRITICAL)

# Create an HNSW index for fast similarity search
safe_create_index(vector_conn, vector_store, "oravs_hnsw")


  ✅ Created index: oravs_hnsw


## Ingesting Research Paper Data


In [10]:
from datasets import load_dataset

MAX_PAPERS = 1000
ds_stream = load_dataset("nick007x/arxiv-papers", split="train", streaming=True)

sampled_papers = []
texts = []
metadata = []

for i, item in enumerate(ds_stream):
    if i >= MAX_PAPERS:
        break

    arxiv_id = item.get("arxiv_id", f"unknown_{i}")
    title = (item.get("title") or "").strip()
    abstract = (item.get("abstract") or "").strip()
    primary_subject = (item.get("primary_subject") or "").strip()
    authors = item.get("authors") or []

    if isinstance(authors, str):
        authors_text = authors
    elif isinstance(authors, list):
        authors_text = ", ".join(str(a).strip() for a in authors if str(a).strip())
    else:
        authors_text = ""

    text = f"Title: {title}\nAbstract: {abstract}"

    sampled_papers.append({
        "arxiv_id": arxiv_id,
        "title": title,
        "abstract": abstract,
        "primary_subject": primary_subject,
        "authors": authors_text,
    })
    texts.append(text)
    metadata.append({
        "id": arxiv_id,
        "arxiv_id": arxiv_id,
        "title": title,
        "primary_subject": primary_subject,
        "authors": authors_text,
    })

vector_store.add_texts(
    texts=texts,
    metadatas=metadata,
)

print(f"✅ Ingested {len(texts)} research papers into VECTOR_SEARCH_DEMO")


✅ Ingested 1000 research papers into VECTOR_SEARCH_DEMO


In [11]:
sample_primary_subject = sampled_papers[0]["primary_subject"] if sampled_papers else ""
sample_arxiv_id = sampled_papers[0]["arxiv_id"] if sampled_papers else ""
print("Sample primary subject:", sample_primary_subject)
print("Sample arxiv_id:", sample_arxiv_id)


Sample primary subject: Earth and Planetary Astrophysics (astro-ph.EP)
Sample arxiv_id: 0902.3253


## Querying Vector-Enabled SQL Tables

Search for documents similar to a natural language query. 

The OracleVS layer converts queries to embeddings and finds the closest matches.


Basic Search

In [12]:
query = "Find research papers about planetary exploration mission planning."

results = vector_store.similarity_search(query, k=3)

for i, doc in enumerate(results, start=1):
    print(f"--- Result {i} ---")
    print("Text:", doc.page_content)
    print("Metadata:", doc.metadata)


--- Result 1 ---
Text: Title: Stellar Aspects of Habitability: Characterizing Target Stars for Terrestrial Planet Search Missions
Abstract: In this paper we present and discuss the criteria for selecting potential target stars suitable for the search for Earth like planets, with a special emphasis on the stellar aspects of habitability. Missions that search for terrestrial exoplanets will explore the presence and habitability of Earth-like exoplanets around several hundred nearby stars, mainly F, G, K, and M stars. The evaluation of the list of potential target systems in order to develop mission concepts for a search for Terrestrial Exoplanets is essential. Using the Darwin All Sky Star Catalogue (DASSC), we discuss the selection criteria, configuration dependent sub-catalogues and the implication of stellar activity for habitability.
Metadata: {'id': '0906.0378', 'arxiv_id': '0906.0378', 'title': 'Stellar Aspects of Habitability: Characterizing Target Stars for Terrestrial Planet Sea

Search With Scores

In [13]:
results = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results:
    print("Score:", score)
    print("Text :", doc.page_content)
    print("Meta :", doc.metadata)
    print("------")

Score: 0.35721080800513283
Text : Title: Stellar Aspects of Habitability: Characterizing Target Stars for Terrestrial Planet Search Missions
Abstract: In this paper we present and discuss the criteria for selecting potential target stars suitable for the search for Earth like planets, with a special emphasis on the stellar aspects of habitability. Missions that search for terrestrial exoplanets will explore the presence and habitability of Earth-like exoplanets around several hundred nearby stars, mainly F, G, K, and M stars. The evaluation of the list of potential target systems in order to develop mission concepts for a search for Terrestrial Exoplanets is essential. Using the Darwin All Sky Star Catalogue (DASSC), we discuss the selection criteria, configuration dependent sub-catalogues and the implication of stellar activity for habitability.
Meta : {'id': '0906.0378', 'arxiv_id': '0906.0378', 'title': 'Stellar Aspects of Habitability: Characterizing Target Stars for Terrestrial Pl

Filter by exact match on a metadata field

In [14]:
query = "Find papers related to mission planning and observational astronomy."

# This returns docs where metadata.primary_subject matches the sampled subject.
docs = vector_store.similarity_search(
    query,
    k=3,
    filter={"primary_subject": {"$eq": sample_primary_subject}},
)

for doc in docs:
    print("Text:", doc.page_content[:120], "...")
    print("Meta:", doc.metadata)
    print("------")


Text: Title: Search for Life on Exoplanets: Toward an International Institutional Coordination
Abstract: Searching for life in ...
Meta: {'id': '0906.1649', 'arxiv_id': '0906.1649', 'title': 'Search for Life on Exoplanets: Toward an International Institutional Coordination', 'primary_subject': 'Earth and Planetary Astrophysics (astro-ph.EP)', 'authors': 'Jean Schneider, Vincent Coude du Foresto, Marc Ollivier'}
------
Text: Title: International Year of Astronomy Invited Review on Exoplanets
Abstract: Just fourteen years ago the Solar System r ...
Meta: {'id': '0903.3059', 'arxiv_id': '0903.3059', 'title': 'International Year of Astronomy Invited Review on Exoplanets', 'primary_subject': 'Earth and Planetary Astrophysics (astro-ph.EP)', 'authors': 'John A. Johnson'}
------
Text: Title: Kepler Science Operations
Abstract: Kepler&#39;s primary mission is a search for earth-size exoplanets in the hab ...
Meta: {'id': '1001.0437', 'arxiv_id': '1001.0437', 'title': 'Kepler Science Operations

Filter by id list ($in)

In [15]:
docs = vector_store.similarity_search(
    query="Explain key themes in this research paper",
    k=5,
    filter={"id": {"$in": [sample_arxiv_id]}},
)

print(docs)


[Document(id='0902.3253', metadata={'id': '0902.3253', 'arxiv_id': '0902.3253', 'title': 'The gravitational wave background from star-massive black hole fly-bys', 'primary_subject': 'Earth and Planetary Astrophysics (astro-ph.EP)', 'authors': 'Silvia Toonen, Clovis Hopman, Marc Freitag'}, page_content='Title: The gravitational wave background from star-massive black hole fly-bys\nAbstract: Stars on eccentric orbits around a massive black hole (MBH) emit bursts of gravitational waves (GWs) at periapse. Such events may be directly resolvable in the Galactic centre. However, if the star does not spiral in, the emitted GWs are not resolvable for extra-galactic MBHs, but constitute a source of background noise. We estimate the power spectrum of this extreme mass ratio burst background (EMBB) and compare it to the anticipated instrumental noise of the Laser Interferometer Space Antenna (LISA). To this end, we model the regions close to a MBH, accounting for mass-segregation, and for processe

# Memory Engineering and Agent Memory
--------



**`Agent Memory`** is the exocortex that augments an LLM—capturing, encoding, storing, linking, and retrieving information beyond the model’s parametric and contextual limits. 
It provides the persistence and structure required for long-horizon reasoning and reliable behaviour.

**`Memory Engineering`** is the scaffolding and control harness that we design to move information optimally and efficiently into, through, and across all components of an AI system(databases, LLMs, applications etc). It ensures that data is captured, transformed, organized, and retrieved in the right way at the right time—so agents can behave reliably, believably, and capabaly.

This is the core section of the notebook where we build a complete **`Memory Manager`** for AI agents. 

Just like humans have different types of memory (short-term, long-term, procedural), AI agents benefit from specialized memory systems.

## Why Memory Engineering Matters

Without memory, agents:
- Forget previous conversations
- Can't learn from past interactions
- Repeat the same mistakes
- Lack context for complex tasks

With proper memory engineering, agents can:
- Maintain context across sessions
- Learn and improve over time
- Access relevant knowledge when needed
- Execute complex multi-step workflows

## Memory Types We'll Implement

| Memory Type | Human Analogy | Purpose | Storage |
|-------------|---------------|---------|---------|
| **Conversational** | Short-term memory | Chat history per thread | SQL Table |
| **Knowledge Base** | Long-term semantic memory | Facts, documents, search results | Vector-Enabled SQL Table |
| **Workflow** | Procedural memory | Learned action patterns | Vector-Enabled SQL Table |
| **Toolbox** | Skill memory | Available tools & capabilities | Vector-Enabled SQL Table |
| **Entity** | Episodic memory | People, places, systems mentioned | Vector-Enabled SQL Table |
| **Summary** | Compressed memory | Condensed context for long conversations | Vector-Enabled SQL Table |

## Steps in This Section

1. **Define table names** for each memory type
2. **Create SQL table** for conversational history
3. **Create vector-enabled SQL tables** for semantic memories
4. **Build indexes** for fast similarity search
5. **Implement MemoryLayer class** with read/write methods for each memory type
6. **Initialize the memory manager** with all storage backends

## Define Memory Tables and Stores
First, we define table names for each memory type. 

These tables will be created in Oracle Database to persist agent memory.

In [16]:
# Table names for each memory type
CONVERSATIONAL_TABLE   = "CONVERSATIONAL_MEMORY" # Episodic memory
KNOWLEDGE_BASE_TABLE   = "SEMANTIC_MEMORY" # Semantic memory
WORKFLOW_TABLE = "WORKFLOW_MEMORY" # Procedural memory
TOOLBOX_TABLE    = "TOOLBOX_MEMORY" # Procedural memory
ENTITY_TABLE = "ENTITY_MEMORY" # Semantic memory
SUMMARY_TABLE = "SUMMARY_MEMORY" # Semanatic memory

ALL_TABLES = [CONVERSATIONAL_TABLE, KNOWLEDGE_BASE_TABLE, WORKFLOW_TABLE, TOOLBOX_TABLE, ENTITY_TABLE, SUMMARY_TABLE]

# Drop existing tables to start fresh
for table in ALL_TABLES:
    try:
        with vector_conn.cursor() as cur:
            cur.execute(f"DROP TABLE {table} PURGE")
    except Exception as e:
        if "ORA-00942" in str(e):
            print(f"  - {table} (not exists)")
        else:
            print(f"  ✗ {table}: {e}")
            
vector_conn.commit()

  - CONVERSATIONAL_MEMORY (not exists)
  - SEMANTIC_MEMORY (not exists)
  - WORKFLOW_MEMORY (not exists)
  - TOOLBOX_MEMORY (not exists)
  - ENTITY_MEMORY (not exists)
  - SUMMARY_MEMORY (not exists)


In [17]:
# Model token limits (for context management)
MODEL_TOKEN_LIMITS = {
    "gpt-5": 256000,
    "gpt-5-mini": 128000,
    "gpt-4o": 128000,
    "gpt-4o-mini": 128000,
    "gpt-4-turbo": 128000,
    "gpt-4": 8192,
    "gpt-3.5-turbo": 16385,
}

### Create Conversational Memory Table

This function below creates a SQL table to store chat history. 

Unlike semantic memories backed by vector-enabled SQL tables, conversational memory uses a traditional table because we need exact retrieval by thread ID (not similarity search).

**What it does:**
- Creates a table with columns: `id`, `thread_id`, `role`, `content`, `timestamp`, `metadata`
- Adds an index on `thread_id` for fast conversation lookups
- Adds an index on `timestamp` for chronological ordering


In [18]:
def create_conversational_history_table(conn, table_name: str = "CONVERSATIONAL_MEMORY"):
    """
    Create a table to store conversational history.

    Args:
        conn: Oracle database connection
        table_name: Name of the table to create
    """
    with conn.cursor() as cur:
        # Drop table if exists
        try:
            cur.execute(f"DROP TABLE {table_name}")
        except:
            pass  # Table doesn't exist
        
        # Create table with proper schema
        cur.execute(f"""
            CREATE TABLE {table_name} (
                id VARCHAR2(100) DEFAULT SYS_GUID() PRIMARY KEY,
                thread_id VARCHAR2(100) NOT NULL,
                role VARCHAR2(50) NOT NULL,
                content CLOB NOT NULL,
                timestamp TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                metadata CLOB,
                created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                summary_id VARCHAR2(100) DEFAULT NULL
            )
        """)
        
        # Create index on thread_id for faster lookups
        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_thread_id ON {table_name}(thread_id)
        """)
        
        # Create index on timestamp for ordering
        cur.execute(f"""
            CREATE INDEX idx_{table_name.lower()}_timestamp ON {table_name}(timestamp)
        """)
        
    conn.commit()
    print(f"Table {table_name} created successfully with indexes")
    return table_name


In [19]:
# Create the table
CONVERSATION_HISTORY_TABLE = create_conversational_history_table(vector_conn, CONVERSATIONAL_TABLE)

Table CONVERSATIONAL_MEMORY created successfully with indexes


### Create Vector-Enabled Tables for Each Memory Type

Here we create 5 separate OracleVS-backed vector-enabled SQL tables—one for each memory type. 

Each semantic memory is backed by its own Oracle table with a VECTOR column and uses the same embedding model for consistency.

| Vector-Enabled Table Handle | Purpose |
|--------------|---------|
| `knowledge_base_vs` | Store documents, facts, and search results |
| `workflow_vs` | Store learned action patterns and tool sequences |
| `toolbox_vs` | Store tool definitions for semantic tool discovery |
| `entity_vs` | Store extracted entities (people, places, systems) |
| `summary_vs` | Store compressed summaries for long conversations |


In [20]:
knowledge_base_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=KNOWLEDGE_BASE_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

workflow_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=WORKFLOW_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

toolbox_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=TOOLBOX_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

entity_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=ENTITY_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)

summary_vs = OracleVS(
    client=vector_conn,
    embedding_function=embedding_model,
    table_name=SUMMARY_TABLE,
    distance_strategy=DistanceStrategy.COSINE,
)


Then we create indexes for each vector-enabled table

In [21]:
print("Creating vector indexes...")
safe_create_index(vector_conn, knowledge_base_vs, "knowledge_base_vs_hnsw")
safe_create_index(vector_conn, workflow_vs, "workflow_vs_hnsw")
safe_create_index(vector_conn, toolbox_vs, "toolbox_vs_hnsw")
safe_create_index(vector_conn, entity_vs, "entity_vs_hnsw")
safe_create_index(vector_conn, summary_vs, "summary_vs_hnsw")
print("All indexes created!")

if "sampled_papers" in globals() and sampled_papers:
    kb_texts = [f"Title: {p['title']}\nAbstract: {p['abstract']}" for p in sampled_papers]
    kb_meta = [
        {
            "id": p["arxiv_id"],
            "arxiv_id": p["arxiv_id"],
            "title": p["title"],
            "primary_subject": p["primary_subject"],
            "authors": p["authors"],
            "source_type": "arxiv_papers",
        }
        for p in sampled_papers
    ]
    knowledge_base_vs.add_texts(kb_texts, kb_meta)
    print(f"✅ Seeded knowledge base memory with {len(kb_texts)} arXiv papers")


Creating vector indexes...
  ✅ Created index: knowledge_base_vs_hnsw
  ✅ Created index: workflow_vs_hnsw
  ✅ Created index: toolbox_vs_hnsw
  ✅ Created index: entity_vs_hnsw
  ✅ Created index: summary_vs_hnsw
All indexes created!
✅ Seeded knowledge base memory with 1000 arXiv papers


## Programmatic vs Agent-Triggered Operations


A key design decision in memory engineering is deciding which operations run **programmatically** (always executed by the harness) versus **agent-triggered** (the LLM chooses to invoke them during the loop).

In this notebook, the harness is intentionally opinionated: memory loading and persistence are automatic, while external/expansion actions are chosen by the agent.

| Operation | Programmatic | Agent-Triggered | Notes |
|-----------|:------------:|:---------------:|-------|
| `read_conversational_memory()` | ✅ | ❌ | Always loaded at loop start (unsummarized units only) |
| `read_knowledge_base()` | ✅ | ❌ | Always loaded at loop start |
| `read_workflow()` | ✅ | ❌ | Always loaded at loop start |
| `read_entity()` | ✅ | ❌ | Always loaded at loop start |
| `read_summary_context()` | ✅ | ❌ | Always loaded at loop start (IDs + descriptions) |
| `read_toolbox()` | ✅ | ❌ | Tool schemas are retrieved before model reasoning |
| `write_conversational_memory()` | ✅ | ❌ | User message (pre-loop) + assistant answer (post-loop) |
| `write_workflow()` | ✅ | ❌ | Persisted after loop when tool steps exist |
| `write_entity()` | ✅ | ❌ | Best-effort extraction around user/final assistant text |
| Tool-call decision (`tool_choice=auto`) | ❌ | ✅ | Model decides whether to call tools |
| `search_tavily()` | ❌ | ✅ | Agent-triggered external retrieval |
| `expand_summary()` | ❌ | ✅ | Agent-triggered just-in-time summary expansion |
| `summarize_and_store()` | ❌ | ✅ | Agent-triggered context compaction primitive |
| `summarize_conversation()` | ❌ | ✅ | Agent-triggered conversation compaction for active thread |

### What Is Programmatic in This Harness

These operations are always executed by code, not delegated to the model:

1. **Context assembly** at the start of `call_agent()`.
2. **Tool schema retrieval** before each model call.
3. **Memory persistence** around the loop (store user turn, store assistant turn, persist workflow/entity updates).
4. **Tool execution dispatch** after a tool call is chosen (once selected by the model, execution is deterministic in code).

### What Is Agent-Triggered in This Harness

These operations are chosen by the model during the loop:

1. **Whether** to call a tool at all.
2. **Which** tool to call.
3. **When** to trigger web search, summary expansion, or conversation compaction.
4. **How** to sequence multiple tool calls before finalizing an answer.

### Why This Split Works for Memory-Centric Agents

1. **Reliability from programmatic memory** — critical memory load/save behavior never depends on the model remembering to do it.
2. **Adaptivity from agent-triggered tools** — the model can selectively fetch/expand/compact only when needed.
3. **Clear control boundaries** — the harness owns state integrity; the model owns strategy inside those boundaries.


## Memory Manager Implementation

The `MemoryManager` class is the central abstraction that unifies all memory operations. It provides a clean interface for reading and writing to different memory types, hiding the complexity of SQL queries and vector-enabled table operations.

### What We're Building

A single class that manages 6 types of memory with consistent read/write patterns:

| Memory Type | Storage | Write Method | Read Method |
|-------------|---------|--------------|-------------|
| **Conversational** | SQL Table | `write_conversational_memory()` | `read_conversational_memory()` |
| **Knowledge Base** | Vector-Enabled SQL Table | `write_knowledge_base()` | `read_knowledge_base()` |
| **Workflow** | Vector-Enabled SQL Table | `write_workflow()` | `read_workflow()` |
| **Toolbox** | Vector-Enabled SQL Table | `write_toolbox()` | `read_toolbox()` |
| **Entity** | Vector-Enabled SQL Table | `write_entity()` | `read_entity()` |
| **Summary** | Vector-Enabled SQL Table | `write_summary()` | `read_summary_memory()`, `read_summary_context()` |

### Key Features

- **Thread-based conversations** — Messages are organized by `thread_id` for multi-conversation support
- **Semantic search** — Vector-enabled SQL tables enable finding relevant content by meaning, not just keywords
- **Metadata filtering** — Workflows filter by `num_steps > 0`, summaries filter by `id`
- **LLM-powered entity extraction** — Automatically extracts people, places, and systems from text
- **Formatted context output** — Each read method returns formatted text ready for the LLM context

### Alternative: Memory Manager Frameworks

There are existing frameworks that abstract memory management for AI agents:

| Framework | Description |
|-----------|-------------|
| **LangChain Memory** | Built-in memory classes (ConversationBufferMemory, VectorStoreRetrieverMemory) |
| **Mem0** | Dedicated memory layer for AI agents with automatic memory management |
| **LlamaIndex** | Document-based memory with various storage backends |
| **Zep** | Long-term memory service for AI assistants |

### Pros and Cons of Building Your Own

| Approach | Pros | Cons |
|----------|------|------|
| **Custom (what we're doing)** | Full control, tailored to your needs, deeper understanding, no external dependencies | More code to maintain, need to handle edge cases yourself |
| **Using a framework** | Faster to implement, battle-tested, community support, handles edge cases | Less control, may not fit your exact use case, additional dependency |

> **For learning purposes**, building your own memory manager (as we do here) gives you a deep understanding of how memory engineering works. 
> 
> **For production**, you might consider using or extending an existing framework. 
>
> For example, this simple notebook only illustrates reads and writes, but not deletion and updates.

In [22]:
import json as json_lib
from datetime import datetime

class MemoryManager:
    """
    A simplified memory manager for AI agents using Oracle AI Database.
    
    Manages 5 types of memory:
    - Conversational: Chat history per thread (SQL table)
    - Knowledge Base: Searchable documents (vector-enabled SQL table)
    - Workflow: Execution patterns (vector-enabled SQL table)
    - Toolbox: Available tools (vector-enabled SQL table)
    - Entity: People, places, systems (vector-enabled SQL table)
    - Summary: Storing compressed context window
    """
    
    def __init__(self, conn, conversation_table: str, knowledge_base_vs, workflow_vs, toolbox_vs, entity_vs, summary_vs):
        self.conn = conn
        self.conversation_table = conversation_table
        self.knowledge_base_vs = knowledge_base_vs
        self.workflow_vs = workflow_vs
        self.toolbox_vs = toolbox_vs
        self.entity_vs = entity_vs
        self.summary_vs = summary_vs
    
    # ==================== CONVERSATIONAL MEMORY (SQL) ====================
    
    def write_conversational_memory(self, content: str, role: str, thread_id: str) -> str:
        """Store a message in conversation history."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            id_var = cur.var(str)
            cur.execute(f"""
                INSERT INTO {self.conversation_table} (thread_id, role, content, metadata, timestamp)
                VALUES (:thread_id, :role, :content, :metadata, CURRENT_TIMESTAMP)
                RETURNING id INTO :id
            """, {"thread_id": thread_id, "role": role, "content": content, "metadata": "{}", "id": id_var})
            record_id = id_var.getvalue()[0] if id_var.getvalue() else None
        self.conn.commit()
        return record_id
    
    def get_unsummarized_messages(self, thread_id: str, limit: int = 100) -> list[dict]:
        """Return unsummarized conversation units for a thread."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            cur.execute(f"""
                SELECT id, role, content, timestamp
                FROM {self.conversation_table}
                WHERE thread_id = :thread_id AND summary_id IS NULL
                ORDER BY timestamp ASC
                FETCH FIRST :limit ROWS ONLY
            """, {"thread_id": thread_id, "limit": limit})
            rows = cur.fetchall()

        return [
            {"id": rid, "role": role, "content": content, "timestamp": ts}
            for rid, role, content, ts in rows
        ]

    def read_conversational_memory(self, thread_id: str, limit: int = 10) -> str:
        """Read unsummarized conversation history for a thread."""
        messages = self.get_unsummarized_messages(thread_id, limit=limit)
        lines = [f"[{m['timestamp'].strftime('%H:%M:%S')}] [{m['role']}] {m['content']}" for m in messages]
        messages_formatted = '\n'.join(lines)
        return f"""## Conversation Memory: This is the conversation history for the current thread
### How to use: Use the conversation history to answer the question

{messages_formatted}"""

    def mark_as_summarized(self, thread_id: str, summary_id: str, message_ids: list[str] | None = None):
        """Mark conversation units as summarized."""
        thread_id = str(thread_id)
        with self.conn.cursor() as cur:
            if message_ids:
                cur.executemany(
                    f"""
                    UPDATE {self.conversation_table}
                    SET summary_id = :summary_id
                    WHERE thread_id = :thread_id AND id = :id AND summary_id IS NULL
                    """,
                    [{"summary_id": summary_id, "thread_id": thread_id, "id": mid} for mid in message_ids],
                )
                count = len(message_ids)
            else:
                cur.execute(f"""
                    UPDATE {self.conversation_table}
                    SET summary_id = :summary_id
                    WHERE thread_id = :thread_id AND summary_id IS NULL
                """, {"summary_id": summary_id, "thread_id": thread_id})
                count = cur.rowcount
        self.conn.commit()
        print(f"  📦 Marked {count} messages as summarized (summary_id: {summary_id})")

    # ==================== KNOWLEDGE BASE (Vector-Enabled SQL Table) ====================
    
    def write_knowledge_base(self, text: str, metadata: dict):
        """Store text in knowledge base with metadata."""
        self.knowledge_base_vs.add_texts([text], [metadata])
    
    def read_knowledge_base(self, query: str, k: int = 3) -> str:
        """Search knowledge base for relevant content."""
        results = self.knowledge_base_vs.similarity_search(query, k=k)
        content = "\n".join([doc.page_content for doc in results])
        return f"""## Knowledge Base Memory: This are general information that is relevant to the question
### How to use: Use the knowledge base as background information that can help answer the question

{content}"""
    
    
    # ==================== WORKFLOW (Vector-Enabled SQL Table) ====================
    
    def write_workflow(self, query: str, steps: list, final_answer: str, success: bool = True):
        """Store a completed workflow pattern for future reference."""
        # Format steps as text
        steps_text = "\n".join([f"Step {i+1}: {s}" for i, s in enumerate(steps)])
        text = f"Query: {query}\nSteps:\n{steps_text}\nAnswer: {final_answer[:200]}"
        
        metadata = {
            "query": query,
            "success": success,
            "num_steps": len(steps),
            "timestamp": datetime.now().isoformat()
        }
        self.workflow_vs.add_texts([text], [metadata])
    
    def read_workflow(self, query: str, k: int = 3) -> str:
        """Search for similar past workflows with at least 1 step."""
        # Filter to only include workflows that have steps (num_steps > 0)
        results = self.workflow_vs.similarity_search(
            query, 
            k=k, 
            filter={"num_steps": {"$gt": 0}}
        )
        if not results:
            return "## Workflow Memory\nNo relevant workflows found."
        content = "\n---\n".join([doc.page_content for doc in results])
        return f"""## Workflow Memory: This are the past workflows that are relevant to the question
### How to use: Use the steps and use them to answer the question, especially when using tools and external sources

{content}"""
    
    # ==================== TOOLBOX (Vector-Enabled SQL Table) ====================
    
    def write_toolbox(self, text: str, metadata: dict):
        """Store a tool definition in the toolbox."""
        self.toolbox_vs.add_texts([text], [metadata])
    
    def read_toolbox(self, query: str, k: int = 3) -> list[dict]:
        """Find relevant tools and return OpenAI-compatible schemas."""
        results = self.toolbox_vs.similarity_search(query, k=k)
        tools = []
        for doc in results:
            meta = doc.metadata
            # Extract parameters from metadata and convert to OpenAI format
            stored_params = meta.get("parameters", {})
            properties = {}
            required = []
            
            for param_name, param_info in stored_params.items():
                # Convert stored param info to OpenAI schema format
                param_type = param_info.get("type", "string")
                # Map Python types to JSON schema types
                type_mapping = {
                    "<class 'str'>": "string",
                    "<class 'int'>": "integer", 
                    "<class 'float'>": "number",
                    "<class 'bool'>": "boolean",
                    "str": "string",
                    "int": "integer",
                    "float": "number",
                    "bool": "boolean"
                }
                json_type = type_mapping.get(param_type, "string")
                properties[param_name] = {"type": json_type}
                
                # If no default, it's required
                if "default" not in param_info:
                    required.append(param_name)
            
            tools.append({
                "type": "function",
                "function": {
                    "name": meta.get("name", "tool"),
                    "description": meta.get("description", ""),
                    "parameters": {"type": "object", "properties": properties, "required": required}
                }
            })
        return tools

    # ==================== ENTITY (Vector-Enabled SQL Table) ====================
    
    def extract_entities(self, text: str, llm_client) -> list[dict]:
        """Use LLM to extract entities (people, places, systems) from text."""
        if not text or len(text.strip()) < 5:
            return []
        
        prompt = f'''Extract entities from: "{text[:500]}"
Return JSON: [{{"name": "X", "type": "PERSON|PLACE|SYSTEM", "description": "brief"}}]
If none: []'''

        try:
            response = llm_client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[{"role": "user", "content": prompt}],
                temperature=0.0,
                max_tokens=300
            )
            result = response.choices[0].message.content.strip()
            
            # Extract JSON array from response
            start, end = result.find("["), result.rfind("]")
            if start == -1 or end == -1:
                return []
            
            parsed = json_lib.loads(result[start:end+1])
            return [{"name": e["name"], "type": e.get("type", "UNKNOWN"), "description": e.get("description", "")} 
                    for e in parsed if isinstance(e, dict) and e.get("name")]
        except:
            return []
    
    def write_entity(self, name: str, entity_type: str, description: str, llm_client=None, text: str = None):
        """Store an entity OR extract and store entities from text."""
        if text and llm_client:
            # Extract and store entities from text
            entities = self.extract_entities(text, llm_client)
            for e in entities:
                self.entity_vs.add_texts(
                    [f"{e['name']} ({e['type']}): {e['description']}"],
                    [{"name": e['name'], "type": e['type'], "description": e['description']}]
                )
            return entities
        else:
            # Store single entity directly
            self.entity_vs.add_texts(
                [f"{name} ({entity_type}): {description}"],
                [{"name": name, "type": entity_type, "description": description}]
            )
    
    def read_entity(self, query: str, k: int = 5) -> str:
        """Search for relevant entities."""
        results = self.entity_vs.similarity_search(query, k=k)
        if not results:
            return "## Entity Memory\nNo entities found."
        
        entities = [f"• {doc.metadata.get('name', '?')}: {doc.metadata.get('description', '')}" 
                    for doc in results if hasattr(doc, 'metadata')]
        entities_formatted = '\n'.join(entities)
        return f"""## Entity Memory: This are the entities that are relevant to the question
### How to use: Use the entities to answer the question, especially when having long conversations

{entities_formatted}"""
    
    # ==================== SUMMARY (Vector-Enabled SQL Table) ====================
    
    def write_summary(self, summary_id: str, full_content: str, summary: str, description: str):
        """Store a summary with its original content."""
        self.summary_vs.add_texts(
            [f"{summary_id}: {description}"],
            [{"id": summary_id, "full_content": full_content, "summary": summary, "description": description}]
        )
        return summary_id
    
    def read_summary_memory(self, summary_id: str) -> str:
        """Retrieve a specific summary by ID (just-in-time retrieval)."""
        results = self.summary_vs.similarity_search(
            summary_id, 
            k=5, 
            filter={"id": summary_id}
        )
        if not results:
            return f"Summary {summary_id} not found."
        doc = results[0]
        return doc.metadata.get('summary', 'No summary content.')
    
    def read_summary_context(self, query: str = "", k: int = 10) -> str:
        """Get available summaries for context window (IDs + descriptions only)."""
        results = self.summary_vs.similarity_search(query or "summary", k=k)
        if not results:
            return "## Summary Memory\nNo summaries available."
        
        lines = ["## Summary Memory", "Use expand_summary(id) to get full content if needed:"]
        for doc in results:
            sid = doc.metadata.get('id', '?')
            desc = doc.metadata.get('description', 'No description')
            lines.append(f"  • [ID: {sid}] {desc}")
        return "\n".join(lines) 

In [23]:
# Initialize the MemoryLayer instance
# Note: Uses SQL table for conversational memory, vector-enabled SQL tables for others
memory_manager = MemoryManager(
    conn=vector_conn,
    conversation_table=CONVERSATION_HISTORY_TABLE, 
    knowledge_base_vs=knowledge_base_vs,
    workflow_vs=workflow_vs,
    toolbox_vs=toolbox_vs,
    entity_vs=entity_vs,
    summary_vs=summary_vs
)

## Creating the Agent's Toolbox

### The Scalability Problem with Tools

As your AI system grows, you might have **hundreds of tools** available—APIs, database queries, calculators, search engines, and more. However, passing all tools to the LLM at inference time creates serious problems:

| Problem | Impact |
|---------|--------|
| **Context bloat** | Tool definitions consume tokens, leaving less room for actual content |
| **Tool selection failure** | LLMs struggle to choose the right tool when presented with too many options |
| **Increased latency** | More tokens = slower inference |
| **Higher costs** | More tokens = higher API costs |

Model providers like OpenAI and Anthropic typically recommend limiting the number of tools exposed to an LLM (often 10-20 max for reliable selection).

### The Solution: Semantic Tool Retrieval

The `Toolbox` class solves this by treating tools as a **searchable memory**:

1. **Register hundreds of tools** — Store all available tools with their descriptions and embeddings
2. **Retrieve only relevant tools** — At inference time, use vector search to find tools semantically relevant to the current query
3. **Pass a focused toolset** — Only the retrieved tools (typically 3-5) are passed to the LLM

This approach means your system can **scale to hundreds of tools** while the LLM only sees the most relevant ones for each query.

### How the Code Works

The `Toolbox` class uses **docstrings as the retrieval key**:

```
User Query → Embed Query → Vector Search → Find tools with similar docstrings → Return relevant tools
```

| Component | Purpose |
|-----------|---------|
| `get_embedding()` | Converts tool description to a vector |
| `ToolMetadata` | Pydantic model storing tool name, description, signature, parameters |
| `_augment_docstring()` | Uses LLM to improve the docstring for better retrieval |
| `_generate_queries()` | Creates synthetic queries that would trigger this tool |
| `register_tool()` | Decorator that stores tool with its embedding in the toolbox |

When you call `memory_manager.read_toolbox(query)`, it performs a similarity search to find tools whose docstrings are semantically similar to the query.

### The Intersection of Three Engineering Disciplines

This implementation combines techniques from **memory engineering**, **context engineering**, and **prompt engineering**:

| Discipline | Technique Used | How It Helps |
|------------|----------------|--------------|
| **Memory Engineering** | Toolbox as procedural memory | Tools are stored and retrieved like learned skills |
| **Memory Engineering** | Docstring augmentation | LLM improves docstrings for better semantic retrieval |
| **Memory Engineering** | Synthetic query generation | Creates example queries to improve tool discoverability |
| **Context Engineering** | Selective tool retrieval | Only relevant tools enter the context, reducing bloat |
| **Context Engineering** | Context offloading | Tool results can be summarized to save context space |
| **Prompt Engineering** | Role setting | "You are a technical writer" improves docstring quality |

### Key Insight

The `augment=True` flag in `@toolbox.register_tool(augment=True)` triggers:
1. **Docstring augmentation** — LLM rewrites the docstring to be clearer and more searchable
2. **Synthetic query generation** — LLM generates example queries that would need this tool
3. **Rich embedding** — Combines name + augmented docstring + signature + queries for better retrieval

This means a simple one-line docstring like `"Search the web"` becomes a rich, detailed description that's much more likely to be retrieved when the user asks something like `"What's the latest news about AI?"`

In [44]:
import inspect
import uuid
from typing import Callable, Optional, Union
from pydantic import BaseModel

def get_embedding(text: str) -> list[float]:
    """
    Get the embedding for a text using the configured embedding model.
    """
    return embedding_model.embed_query(text)


class ToolMetadata(BaseModel):
    """Metadata for a registered tool."""
    name: str
    description: str
    signature: str
    parameters: dict
    return_type: str


class Toolbox:
    """
    A toolbox for registering, storing, and retrieving tools with LLM-powered augmentation.
    
    Tools are stored with embeddings for semantic retrieval, allowing the agent to
    find relevant tools based on natural language queries.
    """
    
    def __init__(self, memory_manager, llm_client, model: str = "gpt-4o-mini"):
        """
        Initialize the Toolbox.
        
        Args:
            memory_manager: MemoryManager instance for storing tools
            llm_client: OpenAI client for LLM augmentation
            model: Model to use for augmentation (default: gpt-4o-mini)
        """
        self.memory_manager = memory_manager
        self.llm_client = llm_client
        self.model = model
        self._tools: dict[str, Callable] = {}  # Maps tool_id -> callable
        self._tools_by_name: dict[str, Callable] = {}  # Maps function_name -> callable for execution
    
    def _augment_docstring(self, docstring: str) -> str:
        """
        Use LLM to improve and expand a tool's docstring.
        
        Takes a basic docstring and returns an enhanced version with:
        - Clearer description of what the tool does
        - Better formatted parameters and return values
        - Usage examples and edge cases
        
        Args:
            docstring: The original docstring to augment
            
        Returns:
            An improved, more detailed docstring
        """
        if not docstring.strip():
            return "No description provided."


        # NOTE: The role description of a technical writer below is a prompt engineering technique that is used to improve the quality of the docstring
        # Athough there are research that suggest that role description doesn't realy affect the quality of the LLM's output, it is still a useful technique
        # and it is a good [prompt engineering] technique to know.
        prompt = f"""You are a technical writer. Improve the following function docstring to be more clear, 
            comprehensive, and useful. Include:
            1. A clear concise summary
            2. Detailed description of what the function does
            3. When to use this function
            4. Any important notes or caveats

            Original docstring:
            {docstring}

            Return ONLY the improved docstring, no other text.
        """

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=500
        )
        
        return response.choices[0].message.content.strip()
    
    def _generate_queries(self, docstring: str, num_queries: int = 5) -> list[str]:
        """
        Generate synthetic example queries that would lead to using this tool.
        
        These queries are used to improve retrieval - by embedding both the tool
        description AND example queries, we increase the chances of finding the
        right tool when the user asks a related question.
        
        Args:
            docstring: The tool's docstring (ideally augmented)
            num_queries: Number of example queries to generate
            
        Returns:
            List of example natural language queries
        """
        prompt = f"""Based on the following tool description, generate {num_queries} diverse example queries 
            that a user might ask when they need this tool. Make them natural and varied.

            Tool description:
            {docstring}

            Return ONLY a JSON array of strings, like: ["query1", "query2", ...]
        """

        response = self.llm_client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.3,
            max_tokens=300
        )
        
        try:
            import json
            queries = json.loads(response.choices[0].message.content.strip())
            return queries if isinstance(queries, list) else []
        except json.JSONDecodeError:
            # Fallback: extract queries from text
            return [response.choices[0].message.content.strip()]
    
    def _get_tool_metadata(self, func: Callable) -> ToolMetadata:
        """
        Extract metadata from a function for storage and retrieval.
        
        Args:
            func: The function to extract metadata from
            
        Returns:
            ToolMetadata object with function details
        """
        sig = inspect.signature(func)
        
        # Extract parameter info
        parameters = {}
        for name, param in sig.parameters.items():
            param_info = {"name": name}
            if param.annotation != inspect.Parameter.empty:
                param_info["type"] = str(param.annotation)
            if param.default != inspect.Parameter.empty:
                param_info["default"] = str(param.default)
            parameters[name] = param_info
        
        # Extract return type
        return_type = "Any"
        if sig.return_annotation != inspect.Signature.empty:
            return_type = str(sig.return_annotation)
        
        return ToolMetadata(
            name=func.__name__,
            description=func.__doc__ or "No description",
            signature=str(sig),
            parameters=parameters,
            return_type=return_type
        )
    
    def register_tool(
        self, func: Optional[Callable] = None, augment: bool = False
    ) -> Union[str, Callable]:
        """
        Register a function as a tool in the toolbox.

        Can be used as a decorator or called directly:
        
            @toolbox.register_tool
            def my_tool(): ...
            
            @toolbox.register_tool(augment=True)
            def my_enhanced_tool(): ...
            
            tool_id = toolbox.register_tool(some_function)

        Parameters:
        -----------
        func : Callable, optional
            The function to register as a tool. If None, returns a decorator.
        augment : bool, optional
            Whether to augment the tool docstring and generate synthetic queries
            using the configured LLM provider.
            
        Returns:
        --------
        Union[str, Callable]
            If func is provided, returns the tool ID. Otherwise returns a decorator.
        """

        def decorator(f: Callable) -> str:
            docstring = f.__doc__ or ""
            signature = str(inspect.signature(f))
            object_id = uuid.uuid4()
            object_id_str = str(object_id)

            # NOTE: Augmentation is a technique that is used to improve the quality of the tool's docstring
            # by using the LLM to enhance the tool's discoverability and retrieval this is a [memory engineering] technique
            if augment:
                # Use LLM to enhance the tool's discoverability
                augmented_docstring = self._augment_docstring(docstring)
                queries = self._generate_queries(augmented_docstring)
                
                # Create rich embedding text combining all information
                embedding_text = f"{f.__name__} {augmented_docstring} {signature} {' '.join(queries)}"
                embedding = get_embedding(embedding_text)
                
                tool_data = self._get_tool_metadata(f)
                tool_data.description = augmented_docstring  # Use augmented description

                tool_dict = {
                    "_id": object_id_str,  # Use string, not UUID object
                    "embedding": embedding,
                    "queries": queries,
                    "augmented": True,
                    **tool_data.model_dump(),
                }
            else:
                # Basic registration without augmentation
                embedding = get_embedding(f"{f.__name__} {docstring} {signature}")
                tool_data = self._get_tool_metadata(f)

                tool_dict = {
                    "_id": object_id_str,  # Use string, not UUID object
                    "embedding": embedding,
                    "augmented": False,
                    **tool_data.model_dump(),
                }

            # Store the tool in the toolbox memory for retrieval
            # The embedding enables semantic search to find relevant tools
            self.memory_manager.write_toolbox(
                f"{f.__name__} {docstring} {signature}", 
                tool_dict
            )
            
            # Keep reference to the callable for execution
            self._tools[object_id_str] = f
            self._tools_by_name[f.__name__] = f  # Also store by name for easy lookup
            return object_id_str

        if func is None:
            return decorator
        return decorator(func)


In [ ]:
# import os
# import getpass
#
# # Function to securely get and set environment variables
# def set_env_securely(var_name, prompt):
#     value = getpass.getpass(prompt)
#     os.environ[var_name] = value
#

In [25]:
import os
import getpass

# Azure-specific helper: we keep this separate so the original API-key helper remains untouched.
# We prompt for non-secret config values (endpoint/deployment) with input(), and secrets with getpass() if needed.
def set_env_securely_azure(var_name, prompt, secret=False):
    if secret:
        value = getpass.getpass(prompt)
    else:
        value = input(prompt).strip()
    os.environ[var_name] = value
    return value

In [ ]:
# set_env_securely("OPENAI_API_KEY", "OpenAI API Key: ")

In [26]:
# Azure uses endpoint + deployment with Entra ID (RBAC), not OPENAI_API_KEY.
set_env_securely_azure(
    "AZURE_OPENAI_ENDPOINT",
    "Azure OpenAI endpoint (e.g. https://<resource>.openai.azure.com): "
)
set_env_securely_azure(
    "AZURE_OPENAI_DEPLOYMENT",
    "Azure OpenAI deployment name (e.g. gpt-4o-mini): "
)

# Responses-compatible API version default (override if your resource requires another version).
os.environ.setdefault("AZURE_OPENAI_API_VERSION", "2025-03-01-preview")
print("RBAC auth enabled: sign in first (for example, az login).")

RBAC auth enabled: sign in first (for example, az login).


In [ ]:
# from openai import OpenAI
#
# client = OpenAI()
#
# # Initialize the Toolbox
# toolbox = Toolbox(memory_manager=memory_manager, llm_client=client)

In [27]:
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

# Azure OpenAI with RBAC: obtain tokens from DefaultAzureCredential instead of API keys.
credential_azure = DefaultAzureCredential()
token_provider_azure = get_bearer_token_provider(
    credential_azure,
    "https://cognitiveservices.azure.com/.default",
)

# Deployment routing in Azure uses resource endpoint + deployment name.
raw_endpoint_azure = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
if "/openai" in raw_endpoint_azure.lower():
    raw_endpoint_azure = raw_endpoint_azure[: raw_endpoint_azure.lower().index("/openai")]

openai_model_azure = os.environ["AZURE_OPENAI_DEPLOYMENT"]
api_version_azure = os.environ.get("AZURE_OPENAI_API_VERSION", "2025-03-01-preview")

client_azure = AzureOpenAI(
    azure_endpoint=raw_endpoint_azure,
    api_version=api_version_azure,
    azure_ad_token_provider=token_provider_azure,
    api_key="azure-ad-token-provider",  # compatibility for openai package credential checks
)

toolbox_azure = Toolbox(memory_manager=memory_manager, llm_client=client_azure, model=openai_model_azure)

# Compatibility aliases so existing downstream cells continue to work unchanged.
client = client_azure
toolbox = toolbox_azure
OPENAI_MODEL = openai_model_azure

print(f"Azure endpoint: {raw_endpoint_azure}")
print(f"Azure deployment: {openai_model_azure}")
print(f"Azure API version: {api_version_azure}")

Azure endpoint: https://azure-agent-ai-foundry-resource.openai.azure.com
Azure deployment: gpt-4.1
Azure API version: 2025-03-01-preview


In [28]:
import json as json_lib

# Azure-specific patch: Azure expects the deployment name, not a fixed base model string.
# We patch only at runtime after PAUSE to keep the original class definition untouched.
def extract_entities_azure(self, text: str, llm_client) -> list[dict]:
    if not text or len(text.strip()) < 5:
        return []

    prompt = f'''Extract entities from: "{text[:500]}"
Return JSON: [{{"name": "X", "type": "PERSON|PLACE|SYSTEM", "description": "brief"}}]
If none: []'''

    try:
        response = llm_client.chat.completions.create(
            model=globals().get("openai_model_azure", "gpt-4o-mini"),
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=300,
        )
        result = response.choices[0].message.content.strip()
        start, end = result.find("["), result.rfind("]")
        if start == -1 or end == -1:
            return []
        parsed = json_lib.loads(result[start:end+1])
        return [
            {
                "name": e["name"],
                "type": e.get("type", "UNKNOWN"),
                "description": e.get("description", ""),
            }
            for e in parsed
            if isinstance(e, dict) and e.get("name")
        ]
    except Exception:
        return []

MemoryManager.extract_entities = extract_entities_azure

# Context Engineering Techniques

--------


> **Context engineering** refers to the set of strategies for curating and maintaining the optimal set of tokens (information) during LLM inference, including all the other information that may land there outside of the prompts.
> 
> — *Anthropic*

While memory engineering focuses on *what to store and retrieve*, context engineering focuses on *how to manage what's in the context window right now*. This includes monitoring usage, compressing information, and providing just-in-time access to details.

## What This Section Covers

| Step | Function | Purpose |
|------|----------|---------|
| **1. Calculate Usage** | `calculate_context_usage()` | Monitor what % of the context window is used |
| **2. Summarize** | `summarise_context_window()` | Compress long content into summaries using LLM |
| **3. Offload** | `offload_to_summary()` | Auto-trigger summarization when usage exceeds threshold |
| **4. Just-in-Time Retrieval** | `expand_summary()` tool | Let agent expand summaries on demand |

**`Just-In-Time (JIT)`** retrieval is the process of fetching only the information needed at the exact moment the agent requires it, based on the current task, query, or reasoning step. Instead of loading pre-computed or pre-cached context upfront, the system dynamically retrieves the minimal, most relevant data on demand, ensuring efficiency and reducing context overload. In the context of agent memory JIT is a retrieval-control strategy where memory access is triggered by the agent’s current goal, query, or reasoning step. Rather than preloading large histories or the full knowledge base, the system dynamically filters, ranks, and injects only the information that materially influences the next token. This reduces context saturation, improves attention allocation, and increases reasoning fidelity.

## The Context Management Flow

```
Context built → Check usage % → If >80%: Summarize & offload → Store summary with ID
                                                              ↓
Agent sees: [Summary ID: abc123] Brief description ← Agent can call expand_summary("abc123") if needed
```

This approach keeps the context lean while giving the agent access to full details when required.

In [29]:
# Context window calculator - returns percentage used
def calculate_context_usage(context: str, model: str = "gpt-4o-mini") -> dict:
    """Calculate context window usage as percentage."""
    estimated_tokens = len(context) // 4  # ~4 chars per token
    max_tokens = MODEL_TOKEN_LIMITS.get(model, 128000)
    percentage = (estimated_tokens / max_tokens) * 100
    return {"tokens": estimated_tokens, "max": max_tokens, "percent": round(percentage, 1)}


In [ ]:
# # Context summariser - calls LLM and stores summary
# import uuid
#
# def summarise_context_window(content: str, memory_manager, llm_client, model: str = "gpt-4o-mini") -> dict:
#     """Summarise context window using LLM and store in summary memory."""
#     summary_prompt = f"""
# You are compressing an AI agent context window for later retrieval.
# The content may include conversation memory, retrieved papers, entities, workflows, and prior summaries.
#
# Produce a compact summary that preserves:
# - user goal and constraints
# - key facts/findings already established
# - important entities (paper titles, arXiv IDs, authors)
# - unresolved questions and next actions
#
# Output 4-7 short bullet points.
# Be faithful to the source, and do not add new facts.
#
# Context window content:
# {content[:3000]}
# """.strip()
#
#     response = llm_client.chat.completions.create(
#         model=model,
#         messages=[{"role": "user", "content": summary_prompt}],
#         max_tokens=220
#     )
#     summary = response.choices[0].message.content
#
#     desc_response = llm_client.chat.completions.create(
#         model=model,
#         messages=[{"role": "user", "content": f"Write a short label (max 12 words) for this summary:\n{summary}"}],
#         max_tokens=40
#     )
#     description = desc_response.choices[0].message.content.strip()
#
#     summary_id = str(uuid.uuid4())[:8]
#     memory_manager.write_summary(summary_id, content, summary, description)
#
#     return {"id": summary_id, "description": description, "summary": summary}
#

In [30]:
# Azure counterpart: explicit _azure function that defaults to Azure deployment naming.
import uuid

def summarise_context_window_azure(content: str, memory_manager, llm_client, model_azure: str = None) -> dict:
    """Summarise context window using Azure OpenAI and store in summary memory."""
    active_model_azure = model_azure or globals().get("openai_model_azure", "gpt-4o-mini")
    summary_prompt = f"""
You are compressing an AI agent context window for later retrieval.
The content may include conversation memory, retrieved papers, entities, workflows, and prior summaries.

Produce a compact summary that preserves:
- user goal and constraints
- key facts/findings already established
- important entities (paper titles, arXiv IDs, authors)
- unresolved questions and next actions

Output 4-7 short bullet points.
Be faithful to the source, and do not add new facts.

Context window content:
{content[:3000]}
""".strip()

    response = llm_client.chat.completions.create(
        model=active_model_azure,
        messages=[{"role": "user", "content": summary_prompt}],
        max_tokens=220
    )
    summary = response.choices[0].message.content

    desc_response = llm_client.chat.completions.create(
        model=active_model_azure,
        messages=[{"role": "user", "content": f"Write a short label (max 12 words) for this summary:\n{summary}"}],
        max_tokens=40
    )
    description = desc_response.choices[0].message.content.strip()

    summary_id = str(uuid.uuid4())[:8]
    memory_manager.write_summary(summary_id, content, summary, description)
    return {"id": summary_id, "description": description, "summary": summary}

# Keep downstream cells unchanged by aliasing the active implementation.
summarise_context_window = summarise_context_window_azure

In [31]:
# Context offloader - replaces content with summary reference
def offload_to_summary(context: str, memory_manager, llm_client, threshold_percent: float = 80.0) -> tuple:
    """If context exceeds threshold, summarise and return compacted version."""
    usage = calculate_context_usage(context)
    
    if usage['percent'] < threshold_percent:
        return context, []  # No offload needed
    
    # Summarise the context
    result = summarise_context_window(context, memory_manager, llm_client)
    
    # Return compact reference instead of full content
    compact = f"[Summary ID: {result['id']}] {result['description']}"
    return compact, [result]


### Summary Tools & Conversation Compaction

Below we register the `expand_summary` and `summarize_and_store` functions as tools the agent can call.

#### Design Logic: Why Mark Instead of Delete?

When conversation history grows large, we need to reduce context window usage. We had two choices:

| Approach | Pros | Cons |
|----------|------|------|
| **Delete summarized messages** | Simple, immediate space savings | Permanent data loss, can't audit or recover |
| **Mark as summarized (our choice)** | Preserves history, reversible, auditable | Slightly more complex queries |

**Our intuition:** Memory should be *compressed*, or *forgotten* not *erased*. By marking messages with a `summary_id` instead of deleting them:

1. **Full history is preserved** — Original messages remain in the database for auditing, debugging, or reprocessing
2. **Linkage is maintained** — Each summary knows which messages it represents (via `summary_id`)
3. **Reversible** — If a summary is deleted, you could "unsummarize" by clearing the `summary_id`

#### The Flow

```
Thread has 50 messages → Context too large → summarize_conversation(thread_id)
                                                    ↓
                        1. Read unsummarized messages
                        2. LLM summarizes them
                        3. Store summary with unique ID
                        4. UPDATE messages SET summary_id = 'abc123'
                                                    ↓
                        Next read: Only new messages appear + Summary ID reference
```

This is a form of **log compaction** — a pattern borrowed from databases and message queues where old entries are compressed but not lost.

In [32]:
# Summary tools for the agent
@toolbox.register_tool(augment=True)
def expand_summary(summary_id: str) -> str:
    """Expand a summary reference to full content. Use when you need more details from a [Summary ID: xxx] reference."""
    return memory_manager.read_summary_memory(summary_id)

@toolbox.register_tool(augment=True)
def summarize_and_store(text: str) -> str:
    """Summarize a long text block and store it. Returns [Summary ID: ...] for later expansion."""
    result = summarise_context_window(text, memory_manager, client)
    return f"Stored as [Summary ID: {result['id']}] {result['description']}"

@toolbox.register_tool(augment=True)
def summarize_conversation(thread_id: str) -> str:
    """
    Summarize unsummarized conversation units for a thread and mark those units with summary_id.
    Use this when conversation memory becomes long and you need context compaction.
    """
    unsummarized = memory_manager.get_unsummarized_messages(thread_id, limit=200)
    if not unsummarized:
        return "No unsummarized conversation units found."

    full_text = "\n".join([f"[{m['role']}] {m['content']}" for m in unsummarized])
    result = summarise_context_window(full_text, memory_manager, client)

    message_ids = [m["id"] for m in unsummarized]
    memory_manager.mark_as_summarized(thread_id, result['id'], message_ids=message_ids)

    return f"Conversation summarized as [Summary ID: {result['id']}] {result['description']}"


# Web Access with Tavily

--------

This section demonstrates how to create an **agentic tool** that the LLM can call to search the web. 

We use [Tavily](https://tavily.com/), an AI-optimized search API designed for LLM applications.

## What This Section Does

1. **Initialize the Tavily client** — Set up the search API with an API key
2. **Register `search_tavily` as a tool** — Use `@toolbox.register_tool(augment=True)` to make it discoverable
3. **Implement the search-and-store pattern** — Results are automatically written to knowledge base memory
4. **Test tool retrieval** — Verify the tool can be found via semantic search

## The Search-and-Store Pattern

One thing to note is that not only do we get external context that is not available to the Agent at execution, but we persists this to the knowledge base memory and the Agent can reuse this information in subsequent iteration.
When the agent calls `search_tavily()`, it doesn't just return results—it **persists them to the knowledge base**:

```
Agent calls search_tavily("latest AI news")
       ↓
Tavily API returns results
       ↓
Each result is written to knowledge_base_vs with metadata (title, URL, timestamp)
       ↓
Future queries can retrieve this information without searching again
```

This pattern means the agent **learns** from its searches. Information discovered once becomes part of the agent's long-term memory, available for future conversations without additional API calls.

In [ ]:
# Original reference (commented out)
# set_env_securely("TAVILY_API_KEY", "Tavily API Key: ")

In [34]:
# Reuse the post-PAUSE secure helper; secret=True keeps Tavily key hidden in input.
set_env_securely_azure("TAVILY_API_KEY", "Tavily API Key: ", secret=True)

'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'

In [35]:
from tavily import TavilyClient
from datetime import datetime

# Don't forget to set your API key!
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@toolbox.register_tool(augment=True)
def search_tavily(query: str, max_results: int = 5):
    """
    Use this function to search the web and store the results in the knowledge base.
    """
    response = tavily_client.search(query=query, max_results=max_results)
    results = response.get("results", [])

    # Write each result to the knowledge base
    for result in results:
        # Create the text content to embed
        text = f"Title: {result.get('title', '')}\nContent: {result.get('content', '')}\nURL: {result.get('url', '')}"
        
        # Create metadata
        metadata = {
            "title": result.get("title", ""),
            "url": result.get("url", ""),
            "score": result.get("score", 0),
            "source_type": "tavily_search",
            "query": query,
            "timestamp": datetime.now().isoformat()
        }
        
        # Write to knowledge base
        memory_manager.write_knowledge_base(text, metadata)

    return results

In [36]:
import pprint
retreived_tools = memory_manager.read_toolbox("Search the internet")
pprint.pprint(retreived_tools)

[{'function': {'description': '"""\n'
                              'Searches the web for relevant information and '
                              'stores the results in the knowledge base.\n'
                              '\n'
                              'This function performs a web search based on '
                              'user-defined queries, retrieves relevant web '
                              'pages or documents, and saves the extracted '
                              "results into the application's knowledge base "
                              'for future reference or processing. It may '
                              'utilize search engine APIs or web scraping '
                              'techniques to gather data.\n'
                              '\n'
                              'Use this function when you need to '
                              'automatically augment your knowledge base with '
                              'up-to-date information from the w

# Agent Execution

--------


This is where everything comes together. We build a complete **agent loop** that integrates all the memory types, context engineering, and tool calling we've implemented.

## What This Section Contains

| Component | Purpose |
|-----------|---------|
| `AGENT_SYSTEM_PROMPT` | Instructions telling the LLM how to use memory and tools |
| `execute_tool()` | Looks up and executes tools from the toolbox by name |
| `call_openai_chat()` | Wrapper for OpenAI Chat Completions API with tool support |
| `call_agent()` | The main agent loop that orchestrates everything |


In [ ]:
# import json as json_lib
#
# client = OpenAI()
#
# # ==================== SYSTEM PROMPT ====================
# AGENT_SYSTEM_PROMPT = """
# System Instructions
# You are a Research Paper Assistant with access to memory and tools.
#
# IMPORTANT: The user's input contains CONTEXT that has already been retrieved for you:
# - Conversation Memory: unsummarized conversation units only
# - Knowledge Base Memory: relevant research papers/documents
# - Summary Memory: compressed summaries with IDs and descriptions
#
# ## Summary Memory Rules
# When you see [Summary ID: xxx] entries, call expand_summary(summary_id) only if more detail is required.
# If conversation memory is getting long or repetitive, you may call summarize_conversation(thread_id) to compact it.
# Use summarization tools at your discretion when they improve context quality.
#
# When answering:
# 1. FIRST, use the context provided in the input
# 2. Expand summary IDs just-in-time when needed
# 3. Use external search tools only if memory context is insufficient
# 4. Keep responses evidence-based and aligned with retrieved research context
# """
#
# def execute_tool(tool_name: str, tool_args: dict) -> str:
#     """Execute a tool by looking it up in the toolbox."""
#
#     if tool_name not in toolbox._tools_by_name:
#         return f"Error: Tool '{tool_name}' not found"
#
#     return str(toolbox._tools_by_name[tool_name](**tool_args) or "Done")
#
# # ==================== OPENAI CHAT FUNCTION ====================
# def call_openai_chat(messages: list, tools: list = None, model: str = "gpt-4o-mini", temperature: float = 0.4):
#     """Call OpenAI Chat Completions API with tools."""
#     kwargs = {"model": model, "messages": messages, "temperature": temperature}
#     if tools:
#         kwargs["tools"] = tools
#         kwargs["tool_choice"] = "auto"
#     return client.chat.completions.create(**kwargs)
#

In [37]:
import json as json_lib

# Azure counterpart: separate _azure symbols while keeping original code untouched in previous cells.
# Why different: Azure routes requests by deployment name and uses AAD token auth via DefaultAzureCredential.
AGENT_SYSTEM_PROMPT_azure = """
# System Instructions
You are a Research Paper Assistant with access to memory and tools.

IMPORTANT: The user's input contains CONTEXT that has already been retrieved for you:
- Conversation Memory: unsummarized conversation units only
- Knowledge Base Memory: relevant research papers/documents
- Summary Memory: compressed summaries with IDs and descriptions

## Summary Memory Rules
When you see [Summary ID: xxx] entries, call expand_summary(summary_id) only if more detail is required.
If conversation memory is getting long or repetitive, you may call summarize_conversation(thread_id) to compact it.
Use summarization tools at your discretion when they improve context quality.

When answering:
1. FIRST, use the context provided in the input
2. Expand summary IDs just-in-time when needed
3. Use external search tools only if memory context is insufficient
4. Keep responses evidence-based and aligned with retrieved research context
"""

def execute_tool_azure(tool_name: str, tool_args: dict) -> str:
    """Execute a tool by looking it up in the Azure toolbox."""
    if tool_name not in toolbox_azure._tools_by_name:
        return f"Error: Tool '{tool_name}' not found"
    return str(toolbox_azure._tools_by_name[tool_name](**tool_args) or "Done")

def call_openai_chat_azure(messages: list, tools: list = None, model_azure: str = None, temperature: float = 0.4):
    """Call Azure OpenAI Chat Completions API with tools."""
    active_model_azure = model_azure or globals().get("openai_model_azure", "gpt-4o-mini")
    kwargs = {"model": active_model_azure, "messages": messages, "temperature": temperature}
    if tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"
    return client_azure.chat.completions.create(**kwargs)

# Alias to keep existing call_agent implementation intact.
AGENT_SYSTEM_PROMPT = AGENT_SYSTEM_PROMPT_azure
execute_tool = execute_tool_azure
call_openai_chat = call_openai_chat_azure

## The Agent Loop Flow

```
1. BUILD CONTEXT
   ├── Read conversational memory (chat history)
   ├── Read knowledge base (relevant documents)
   ├── Read workflow memory (past action patterns)
   ├── Read entity memory (people, places, systems)
   └── Read summary context (available summary IDs)

2. CHECK CONTEXT USAGE
   └── If >80% used → Summarize and offload

3. GET TOOLS
   └── Retrieve semantically relevant tools from toolbox

4. STORE USER MESSAGE
   └── Write to conversational memory + extract entities

5. AGENT LOOP (up to max_iterations)
   ├── Call LLM with context + tools
   ├── If tool calls → Execute tools, add results to messages
   └── If no tool calls → Return final answer

6. SAVE RESULTS
   ├── Write workflow (if tools were used)
   ├── Extract entities from response
   └── Store assistant response in conversational memory
```

## Key Design Decisions

- **Memory is loaded programmatically** — The agent always has context without deciding to "remember"
- **Tools are retrieved semantically** — Only relevant tools are passed to the LLM
- **Context is monitored** — Auto-summarization prevents overflow
- **Everything is persisted** — Conversations, workflows, and entities are saved for future use

# PAUSE

In [45]:
# ==================== MAIN AGENT LOOP ====================
def call_agent(query: str, thread_id: str = "1", max_iterations: int = 10) -> str:
    """Agent loop with context window monitoring and tool-driven summarization."""
    thread_id = str(thread_id)
    steps = []

    # 1. Build context from memory
    print("\n" + "="*50)
    print("🧠 BUILDING CONTEXT...")

    context = f"# Question\n{query}\n\n"
    context += memory_manager.read_conversational_memory(thread_id) + "\n\n"
    context += memory_manager.read_knowledge_base(query) + "\n\n"
    context += memory_manager.read_workflow(query) + "\n\n"
    context += memory_manager.read_entity(query) + "\n\n"
    context += memory_manager.read_summary_context(query) + "\n\n"  # IDs + descriptions only

    print("====CONTEXT WINDOW=====\n")
    print(context)

    # 2. Check context usage (agent decides whether to summarize via tools)
    usage = calculate_context_usage(context)
    print(f"📊 Context: {usage['percent']}% ({usage['tokens']}/{usage['max']} tokens)")
    if usage['percent'] > 80:
        print("⚠️ Context >80% - agent may call summarize_conversation(thread_id) for compaction.")

    # 3. Get tools
    dynamic_tools = memory_manager.read_toolbox(query, k=5)

    # Ensure summary tools are available for discretionary compaction/JIT expansion
    summary_tool_candidates = memory_manager.read_toolbox(
        "summarize conversation compact context expand summary memory", k=5
    )
    must_have = {"expand_summary", "summarize_conversation", "summarize_and_store"}
    existing = {t.get("function", {}).get("name") for t in dynamic_tools}

    for tool in summary_tool_candidates:
        name = tool.get("function", {}).get("name")
        if name in must_have and name not in existing:
            dynamic_tools.append(tool)
            existing.add(name)

    print(f"🔧 Tools: {[t['function']['name'] for t in dynamic_tools]}")

    # 4. Store user message & extract entities
    memory_manager.write_conversational_memory(query, "user", thread_id)
    try:
        memory_manager.write_entity("", "", "", llm_client=client, text=query)
    except:
        pass

    # 5. Agent loop
    messages = [{"role": "system", "content": AGENT_SYSTEM_PROMPT}, {"role": "user", "content": context}]
    final_answer = ""

    print("\n🤖 AGENT LOOP")
    for iteration in range(max_iterations):
        print(f"\n--- Iteration {iteration + 1} ---")

        response = call_openai_chat(messages, tools=dynamic_tools)
        msg = response.choices[0].message

        if msg.tool_calls:
            messages.append({"role": "assistant", "content": msg.content, "tool_calls": [
                {"id": tc.id, "type": "function", "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ]})

            for tc in msg.tool_calls:
                tool_name = tc.function.name
                tool_args = json_lib.loads(tc.function.arguments)

                # Ensure conversation compaction always targets the active thread.
                if tool_name == "summarize_conversation":
                    tool_args["thread_id"] = thread_id

                args_display = {k: (v[:50] + '...' if isinstance(v, str) and len(v) > 50 else v)
                               for k, v in tool_args.items()}
                print(f"🛠️ {tool_name}({args_display})")

                try:
                    result = execute_tool(tool_name, tool_args)
                    steps.append(f"{tool_name}({args_display}) → success")
                except Exception as e:
                    result = f"Error: {e}"
                    steps.append(f"{tool_name}({args_display}) → failed")

                print(f"   → {result[:200]}...")
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})
        else:
            final_answer = msg.content or ""
            print(f"\n✅ DONE ({len(steps)} tool calls)")
            break

    # 6. Save workflow & entities
    if steps:
        memory_manager.write_workflow(query, steps, final_answer)
    try:
        memory_manager.write_entity("", "", "", llm_client=client, text=final_answer)
    except:
        pass
    memory_manager.write_conversational_memory(final_answer, "assistant", thread_id)

    print("\n" + "="*50 + f"\n💬 ANSWER:\n{final_answer}\n" + "="*50)
    return final_answer


In [46]:
call_agent("What was my firt question", thread_id="0")



🧠 BUILDING CONTEXT...
====CONTEXT WINDOW=====

# Question
What was my firt question

## Conversation Memory: This is the conversation history for the current thread
### How to use: Use the conversation history to answer the question

[17:12:07] [user] What was my firt question
[17:12:10] [assistant] Your first question in this thread was: "What was my first question?"
[17:13:13] [user] What is Messi?
[17:13:19] [assistant] "Messi" most commonly refers to Lionel Messi, an Argentine professional football (soccer) player widely regarded as one of the greatest players in the history of the sport. He has won numerous awards, including multiple Ballon d'Or titles, and has played for clubs such as FC Barcelona, Paris Saint-Germain, and the Argentina national team. Messi is celebrated for his exceptional skill, vision, and goal-scoring ability.

If you were referring to something else by "Messi," feel free to clarify!
[17:13:26] [user] What is his current club? Is he a potential MVP for the B

'Your first question in this thread was: "What was my first question?"'

In [46]:
call_agent("What is his current club? Is he a potential MVP for the Brazil's national team?", thread_id="0")



🧠 BUILDING CONTEXT...
====CONTEXT WINDOW=====

# Question
What is his current club? Is he a potential MVP for the Brazil's national team?

## Conversation Memory: This is the conversation history for the current thread
### How to use: Use the conversation history to answer the question

[21:13:49] [user] What was my firt question
[21:13:52] [assistant] Your first question in this conversation was:

What was my firt question

This is the only question so far in the conversation history. If you intended to refer to an earlier question, please clarify or provide more context.
[21:14:58] [user] What was my firt question
[21:15:02] [assistant] Your first question in this conversation was:

What was my firt question

This is the only question so far in the conversation history. If you meant a different question or are referring to another thread, please clarify.
[21:15:19] [user] What did we discussed so far?
[21:15:22] [assistant] Here’s a summary of what we discussed so far in this thread

"To answer your questions:\n\n1. What is his current club?\nBased on the entities in memory, you are likely referring to Neymar, the famous Brazilian footballer. As of June 2024, Neymar's current club is Al Hilal, a Saudi Arabian football club. He transferred to Al Hilal from Paris Saint-Germain (PSG) in August 2023.\n\n2. Is he a potential MVP for Brazil's national team?\nNeymar remains one of the most talented and influential players for the Brazilian national team. He is widely regarded as a potential MVP (Most Valuable Player) due to his skill, experience, and leadership. When fit, Neymar is often the focal point of Brazil’s attack and has the ability to change games with his creativity and goal-scoring prowess. However, his MVP status can depend on his health and form during major tournaments.\n\nIf you meant a different player, please clarify the name. Otherwise, these answers are based on Neymar's current status and reputation."

In [47]:
call_agent("What can you tell me about the world cup 2026? Find the recent news in the internet", thread_id="0")



🧠 BUILDING CONTEXT...
====CONTEXT WINDOW=====

# Question
What can you tell me about the world cup 2026? Find the recent news in the internet

## Conversation Memory: This is the conversation history for the current thread
### How to use: Use the conversation history to answer the question

[17:12:07] [user] What was my firt question
[17:12:10] [assistant] Your first question in this thread was: "What was my first question?"
[17:13:13] [user] What is Messi?
[17:13:19] [assistant] "Messi" most commonly refers to Lionel Messi, an Argentine professional football (soccer) player widely regarded as one of the greatest players in the history of the sport. He has won numerous awards, including multiple Ballon d'Or titles, and has played for clubs such as FC Barcelona, Paris Saint-Germain, and the Argentina national team. Messi is celebrated for his exceptional skill, vision, and goal-scoring ability.

If you were referring to something else by "Messi," feel free to clarify!
[17:13:26] [user]

"Here are some recent updates and resources about the FIFA World Cup 2026:\n\n1. **NBC News**: Provides live updates on matches such as Mexico's 1-0 victory over South Korea and Canada's 6-0 win against Qatar. [Read more here](https://www.nbcnews.com/sports/world-cup).\n\n2. **FIFA Official Website**: Offers the latest news, interviews, stats, fixtures, and results for the FIFA World Cup 2026. [Visit FIFA.com](https://www.fifa.com/en/tournaments/mens/worldcup/canadamexicousa2026/news).\n\n3. **World Cup Atlanta 2026**: Atlanta is one of the host cities for the tournament, with the kickoff scheduled for June 15, 2026. [Learn more on the official site](https://atlantafwc26.com).\n\n4. **BBC Sport**: Highlights Mexico's qualification for the last 32 and Canada's dominant performance against Qatar. [Explore details on BBC](https://www.bbc.com/sport/football/world-cup).\n\n5. **YouTube Coverage**: Features live coverage of matches like Mexico vs. South Korea, along with other World Cup-rela

In [48]:
call_agent("Summarize this conversation", thread_id="0")



🧠 BUILDING CONTEXT...
====CONTEXT WINDOW=====

# Question
Summarize this conversation

## Conversation Memory: This is the conversation history for the current thread
### How to use: Use the conversation history to answer the question

[17:12:07] [user] What was my firt question
[17:12:10] [assistant] Your first question in this thread was: "What was my first question?"
[17:13:13] [user] What is Messi?
[17:13:19] [assistant] "Messi" most commonly refers to Lionel Messi, an Argentine professional football (soccer) player widely regarded as one of the greatest players in the history of the sport. He has won numerous awards, including multiple Ballon d'Or titles, and has played for clubs such as FC Barcelona, Paris Saint-Germain, and the Argentina national team. Messi is celebrated for his exceptional skill, vision, and goal-scoring ability.

If you were referring to something else by "Messi," feel free to clarify!
[17:13:26] [user] What is his current club? Is he a potential MVP for the

'The conversation has been summarized as: [Summary ID: 0d0a7f29] "Messi\'s Career & 2026 World Cup Updates: Latest News & Clarifications." Let me know if you\'d like to expand this summary for more details!'

In [50]:
import re
import pandas as pd
from IPython.display import display

if "vector_conn" not in globals() or vector_conn is None:
    raise RuntimeError("Oracle connection `vector_conn` is not available. Run the database setup cells first.")

def _safe_table_name(name: str) -> bool:
    # Allow Oracle identifier characters to avoid SQL injection when building dynamic SQL for table names.
    return bool(re.fullmatch(r"[A-Za-z0-9_$#]+", name or ""))

def _discover_user_tables(conn):
    with conn.cursor() as cur_local:
        cur_local.execute("SELECT table_name FROM user_tables ORDER BY table_name")
        return [row[0] for row in cur_local.fetchall()]

table_names = ALL_TABLES if isinstance(globals().get("ALL_TABLES"), list) and ALL_TABLES else _discover_user_tables(vector_conn)
print(f"Found {len(table_names)} table(s).")

with vector_conn.cursor() as cur_local:
    for table_name in table_names:
        print("\n" + "=" * 80)
        print(f"TABLE: {table_name}")
        print("-" * 80)

        if not _safe_table_name(table_name):
            print("Skipped: table name contains unsupported characters.")
            continue

        try:
            cur_local.execute(f"SELECT * FROM {table_name} FETCH FIRST 20 ROWS ONLY")
            rows = cur_local.fetchall()
            columns = [desc[0] for desc in (cur_local.description or [])]

            df = pd.DataFrame(rows, columns=columns)
            print(f"Rows returned: {len(df)}")
            print(f"Columns ({len(columns)}): {columns}")

            if df.empty:
                print("(No rows in this table)")
            else:
                display(df)
        except Exception as exc:
            print(f"Query failed for {table_name}: {exc}")

Found 6 table(s).

TABLE: CONVERSATIONAL_MEMORY
--------------------------------------------------------------------------------
Rows returned: 14
Columns (8): ['ID', 'THREAD_ID', 'ROLE', 'CONTENT', 'TIMESTAMP', 'METADATA', 'CREATED_AT', 'SUMMARY_ID']


,ID,THREAD_ID,ROLE,CONTENT,TIMESTAMP,METADATA,CREATED_AT,SUMMARY_ID
0,54A0E30C36650E45E063020011AC80FB,0,user,What was my firt question,2026-06-19 17:12:07.112241,{},2026-06-19 17:12:07.112241,0d0a7f29
1,54A0E30C36660E45E063020011AC80FB,0,assistant,"Your first question in this thread was: ""What ...",2026-06-19 17:12:10.917035,{},2026-06-19 17:12:10.917035,0d0a7f29
2,54A0E30C36670E45E063020011AC80FB,0,user,What is Messi?,2026-06-19 17:13:13.457729,{},2026-06-19 17:13:13.457729,0d0a7f29
3,54A0E30C36680E45E063020011AC80FB,0,assistant,"""Messi"" most commonly refers to Lionel Messi, ...",2026-06-19 17:13:19.056096,{},2026-06-19 17:13:19.056096,0d0a7f29
4,54A0E30C36690E45E063020011AC80FB,0,user,What is his current club? Is he a potential MV...,2026-06-19 17:13:26.782132,{},2026-06-19 17:13:26.782132,0d0a7f29
5,54A0E30C366A0E45E063020011AC80FB,0,assistant,Your question seems to be about Lionel Messi. ...,2026-06-19 17:13:33.752546,{},2026-06-19 17:13:33.752546,0d0a7f29
6,54A0E30C366B0E45E063020011AC80FB,0,user,What was his last game and how did it go?,2026-06-19 17:15:11.632553,{},2026-06-19 17:15:11.632553,0d0a7f29
7,54A0E30C366C0E45E063020011AC80FB,0,assistant,Your question is about Lionel Messi's last gam...,2026-06-19 17:15:16.393662,{},2026-06-19 17:15:16.393662,0d0a7f29
8,54A0E30C366D0E45E063020011AC80FB,0,user,What was my firt question,2026-06-19 17:18:29.483330,{},2026-06-19 17:18:29.483330,0d0a7f29
9,54A0E30C366E0E45E063020011AC80FB,0,assistant,"Your first question in this thread was: ""What ...",2026-06-19 17:18:35.767026,{},2026-06-19 17:18:35.767026,0d0a7f29



TABLE: SEMANTIC_MEMORY
--------------------------------------------------------------------------------
Rows returned: 20
Columns (4): ['ID', 'TEXT', 'METADATA', 'EMBEDDING']


,ID,TEXT,METADATA,EMBEDDING
0,b'\xb7q\x05\xdc\xd7A\xfb\x1d',Title: The gravitational wave background from ...,"{'id': '0902.3253', 'arxiv_id': '0902.3253', '...","[-0.13254688680171967, -0.008622904308140278, ..."
1,b';[\xfd\x01\xbc\xd3Q)',Title: Falling Transiting Extrasolar Giant Pla...,"{'id': '0901.2048', 'arxiv_id': '0901.2048', '...","[0.027670102193951607, -0.07764909416437149, 0..."
2,b'\x02\x9e\xbd\x95\x98\xed\xc8\x9d',Title: Dynamics of planets in retrograde mean ...,"{'id': '0902.0428', 'arxiv_id': '0902.0428', '...","[0.10354360193014145, -0.1320883333683014, 0.0..."
3,b'\xbavr\xbf\xd9y\xe1H',Title: Diurnal Thermal Tides in a Non-synchron...,"{'id': '0901.3401', 'arxiv_id': '0901.3401', '...","[0.005644915625452995, -0.1302178055047989, 0...."
4,b'+S\xc2\xd9\xeey<\xe8',"Title: Intermittent turbulence, noisy fluctuat...","{'id': '0901.1570', 'arxiv_id': '0901.1570', '...","[-0.0375816784799099, -0.18754546344280243, -0..."
5,b'\xcfU\xf8\xd3j\xe3\x01\xcf',Title: Dependence of solar wind power spectra ...,"{'id': '0901.4940', 'arxiv_id': '0901.4940', '...","[-0.12204340845346451, -0.0651976615190506, 0...."
6,b'[\xb7t\x1b\xd2\xcb\xf8\xd5',Title: Mercury&#39;s capture into the 3/2 spin...,"{'id': '0901.1843', 'arxiv_id': '0901.1843', '...","[-0.09320578724145889, -0.13648349046707153, -..."
7,b'\xe8l\xa4\x12W0\xf4\xec',Title: Long-term impact risk for (101955) 1999...,"{'id': '0901.3631', 'arxiv_id': '0901.3631', '...","[-0.04725808650255203, -0.13506582379341125, 0..."
8,b'\xe8\xd6\xb5)gn\x8c\xff',Title: HAT-P-11b: A Super-Neptune Planet Trans...,"{'id': '0901.0282', 'arxiv_id': '0901.0282', '...","[-0.0070185186341404915, -0.20573341846466064,..."
9,b'\x9f\x0f\xa2>\x08\x9e\xf1-',Title: Thermal Tides in Short Period Exoplanet...,"{'id': '0901.0735', 'arxiv_id': '0901.0735', '...","[0.0264133233577013, -0.21663077175617218, 0.0..."



TABLE: WORKFLOW_MEMORY
--------------------------------------------------------------------------------
Rows returned: 2
Columns (4): ['ID', 'TEXT', 'METADATA', 'EMBEDDING']


...

,ID,TEXT,METADATA,EMBEDDING
0,b'\xd79x\x1fN\xd8H\x1f',expand_summary Expand a summary reference to f...,{'_id': '4255c84e-19a3-4829-93ec-ca6efc98d5e5'...,"[-0.08031883835792542, 0.08734051138162613, -0..."
1,b'\x89\xd6\xab\x18\xf4gh\x13',summarize_and_store Summarize a long text bloc...,{'_id': 'dcba75c2-02ca-4901-a5e3-2cc8570d167b'...,"[-0.10422181338071823, 0.19213323295116425, -0..."
2,b'\x91fq\x95\xb9}xy',summarize_conversation \n Summarize unsumma...,{'_id': '56a1cc36-4682-48c1-acfc-84ded6d89ae9'...,"[-0.174643874168396, 0.07704905420541763, -0.0..."
3,b'\xf5b\x8fF\xd0\xd4\x88=',search_tavily \n Use this function to searc...,{'_id': 'e9ca88a7-e3bf-469f-ab4e-6be077d73353'...,"[0.008076322264969349, -0.013054369017481804, ..."



TABLE: ENTITY_MEMORY
--------------------------------------------------------------------------------
Rows returned: 14
Columns (4): ['ID', 'TEXT', 'METADATA', 'EMBEDDING']


,ID,TEXT,METADATA,EMBEDDING
0,b'\xc4\x8b\x1d\xa3r\x94\x8cv',Lionel Messi (PERSON): An Argentine profession...,"{'name': 'Lionel Messi', 'type': 'PERSON', 'de...","[0.016081413254141808, -0.13974051177501678, -..."
1,b'A\xc8\x0b=\xe4\xe6\x9f\x1e',Ballon d'Or (SYSTEM): An annual football award...,"{'name': 'Ballon d'Or', 'type': 'SYSTEM', 'des...","[0.10751484334468842, -0.13480818271636963, -0..."
2,"b'v:^\x90\x83""\xc9\xe7'",Lionel Messi (PERSON): A professional football...,"{'name': 'Lionel Messi', 'type': 'PERSON', 'de...","[0.08741569519042969, -0.1301504224538803, 0.0..."
3,b'\xf0\xbd\xa4\x11l\xc8e\x95',FC Barcelona (PLACE): A professional football ...,"{'name': 'FC Barcelona', 'type': 'PLACE', 'des...","[0.01238356065005064, -0.204660102725029, 0.00..."
4,b'\xc5\xbf\xb8\x99\x8e\xd2\x80!',Paris Saint-Germain (PLACE): A professional fo...,"{'name': 'Paris Saint-Germain', 'type': 'PLACE...","[-0.0038082008250057697, -0.07694126665592194,..."
5,b'Z7\x1aS\xd5zR\n',Argentina national team (PLACE): The national ...,"{'name': 'Argentina national team', 'type': 'P...","[0.011748107150197029, -0.06275888532400131, -..."
6,b'\x1b\xfd\xe0\x1f\xe1trs',world cup 2026 (EVENT): An international footb...,"{'name': 'world cup 2026', 'type': 'EVENT', 'd...","[0.06886062771081924, 0.051134418696165085, 0...."
7,"b'\x84\x92\x9f\xc9\xd6\x18""?'",internet (SYSTEM): A global network used to ac...,"{'name': 'internet', 'type': 'SYSTEM', 'descri...","[0.0117426086217165, -0.006943295709788799, -0..."
8,b'g\x90\xe9M\xa4\xd1\xf2\xcb',Messi (PERSON): A professional football player...,"{'name': 'Messi', 'type': 'PERSON', 'descripti...","[0.02817346155643463, -0.1296050250530243, 0.0..."
9,b'\x0c\x88.`7@j;',2026 World Cup (EVENT): An international footb...,"{'name': '2026 World Cup', 'type': 'EVENT', 'd...","[0.07918679714202881, -0.059416282922029495, 0..."



TABLE: SUMMARY_MEMORY
--------------------------------------------------------------------------------
Rows returned: 1
Columns (4): ['ID', 'TEXT', 'METADATA', 'EMBEDDING']


,ID,TEXT,METADATA,EMBEDDING
0,b'\xb6\x02\x9b\xaa\x16\x96z\xe2',"0d0a7f29: ""Messi's Career & 2026 World Cup Upd...","{'id': '0d0a7f29', 'full_content': '[user] Wha...","[-0.021182401105761528, -0.024629253894090652,..."


In [51]:
call_agent("What have we talked about so far?", thread_id="0")



🧠 BUILDING CONTEXT...
====CONTEXT WINDOW=====

# Question
What have we talked about so far?

## Conversation Memory: This is the conversation history for the current thread
### How to use: Use the conversation history to answer the question

[17:26:02] [assistant] The conversation has been summarized as: [Summary ID: 0d0a7f29] "Messi's Career & 2026 World Cup Updates: Latest News & Clarifications." Let me know if you'd like to expand this summary for more details!

## Knowledge Base Memory: This are general information that is relevant to the question
### How to use: Use the knowledge base as background information that can help answer the question

Title: Fermi&#39;s Paradox - The Last Challenge for Copernicanism?
Abstract: We review Fermi&#39;s paradox (or the &#34;Great Silence&#34; problem), not only arguably the oldest and crucial problem for the Search for ExtraTerrestrial Intelligence (SETI), but also a conundrum of profound scientific, philosophical and cultural importance. By

'So far, our conversation has been summarized as discussing "Messi\'s Career & 2026 World Cup Updates: Latest News & Clarifications." Let me know if you\'d like me to expand this summary or provide further details about any specific aspect!'

In [52]:
call_agent("Please expand this summary", thread_id="0")



🧠 BUILDING CONTEXT...
====CONTEXT WINDOW=====

# Question
Please expand this summary

## Conversation Memory: This is the conversation history for the current thread
### How to use: Use the conversation history to answer the question

[17:26:02] [assistant] The conversation has been summarized as: [Summary ID: 0d0a7f29] "Messi's Career & 2026 World Cup Updates: Latest News & Clarifications." Let me know if you'd like to expand this summary for more details!
[17:32:25] [user] What have we talked about so far?
[17:32:29] [assistant] So far, our conversation has been summarized as discussing "Messi's Career & 2026 World Cup Updates: Latest News & Clarifications." Let me know if you'd like me to expand this summary or provide further details about any specific aspect!

## Knowledge Base Memory: This are general information that is relevant to the question
### How to use: Use the knowledge base as background information that can help answer the question

Title: Icy Bodies in the New Solar 

"The expanded summary provides detailed information about Lionel Messi's career and updates on the FIFA World Cup 2026:\n\n1. **Lionel Messi's Career**:\n   - Messi is currently playing for **Inter Miami CF** in Major League Soccer (MLS).\n   - He represents **Argentina** in international football, where he has achieved significant success, including winning the 2022 FIFA World Cup. He is not eligible to play for Brazil's national team.\n\n2. **FIFA World Cup 2026 Updates**:\n   - The tournament will kick off on **June 15, 2026**, with host cities including Atlanta.\n   - Recent matches include Mexico's 1-0 victory over South Korea and Canada's 6-0 win against Qatar.\n   - Relevant updates and coverage can be found on platforms like FIFA.com, NBC News, BBC Sport, and YouTube. \n\nLet me know if you'd like further clarification or additional details!"

## Key Learning for AI Developers: Agent Loop vs Memory-Based Agent Harness

An **agent loop** is the repeated execution cycle: **build context → reason → call tools → observe results → continue until final answer**.

In this notebook, that loop is effective because it is grounded in memory.

An **agent harness** is the full runtime scaffolding around that loop. Here, it is a **memory-based agent harness** where:
- context is assembled from multiple memory types each turn
- tools are discovered and executed in-loop
- outputs are written back into memory for future turns
- summaries compact context while preserving continuity

The key discipline is **context and memory engineering**:
- decide what should be stored, retrieved, summarized, and reused
- keep context windows relevant, not just large
- treat memory as an evolving system that improves agent reliability over time

The practical takeaway: strong agents are not just model prompts. They are loop + harness systems, and memory engineering is the control layer that makes them reliable, stateful, and scalable.
